# P22: Indoor Camera Relocalisation — NetVLAD retrieval vs ACE on 7-Scenes (Chess, Stairs)

**Single-notebook implementation.** Data and pose preparation, metrics and negative controls, NetVLAD retrieval (top-1 and top-k), ACE scene-coordinate regression, the distance-binned comparison with usable range, block-bootstrap statistics, training-density experiments, figures and failure analysis. Run the cells top to bottom; every expensive step is cached and resume-safe.

**Scientific-integrity rules enforced by the code.** Every check records `PASS`, `FAIL` or `NOT_RUN` (with the reason) in a central `STATUS` registry; nothing is ever promoted to `PASS` by default. Published numbers appear only inside the constant `PUBLISHED_REFERENCE`, are labelled as such and are never mixed into measured tables. Hard test frames are never removed.

## 1. Project overview and research objectives

Given one RGB image, estimate the 6-DoF camera pose in a mapped room. Compared: **NetVLAD retrieval** (top-1 and top-k pose interpolation) and **ACE** scene-coordinate regression (RANSAC pose solver). Benchmark: Microsoft 7-Scenes, scenes **Chess** (easier) and **Stairs** (repetitive structure), official train/test sequence splits, two ground-truth variants (D-SLAM and SfM pseudo ground truth).

Required outcome: median translation (cm) and rotation (°) error, both methods, both scenes, both ground truths, and error versus distance to the nearest training view. Extended objective: how far can a query move from the nearest mapped view before localisation becomes unreliable.

*Pre-declared hypotheses (frozen, not to be edited after results exist)*

* **H1** NetVLAD top-1: Spearman correlation(distance, translation error) ≥ 0.5 in both scenes.
* **H2** ACE has a lower such correlation than NetVLAD top-1 in both scenes.
* **H3** Chess, 100 % → 50 % density: NetVLAD top-1 median error grows by a larger factor than ACE's.

## 2. Environment setup and dependency installation

*What:* detect the platform (Colab / Kaggle / other), CUDA, GPU memory and RAM; install only the light dependencies the data and metric sections need. *Why:* those sections are pure NumPy/SciPy; the heavy stack (PyTorch, ACE C++ extension, hloc) is installed in the sections that use it. *Expected output:* an environment report; no failure if no GPU exists.

In [ ]:
import os, sys, json, platform, subprocess, importlib, importlib.util, time, random, math, hashlib, shutil, inspect, itertools, warnings, textwrap, zipfile, urllib.request, urllib.error
from pathlib import Path

IN_KAGGLE = ('KAGGLE_KERNEL_RUN_TYPE' in os.environ) or Path('/kaggle/working').exists()      # checked first: Kaggle images can also contain a google.colab module
IN_COLAB  = (not IN_KAGGLE) and (('google.colab' in sys.modules) or ('COLAB_RELEASE_TAG' in os.environ))
PLATFORM  = 'colab' if IN_COLAB else 'kaggle' if IN_KAGGLE else 'other'

def _pip(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

_REQUIRED = {'numpy': 'numpy', 'scipy': 'scipy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
             'PIL': 'pillow', 'joblib': 'joblib'}
_missing = [pip for mod, pip in _REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print('installing:', _missing); _pip(_missing)

import numpy as np, scipy, pandas as pd, matplotlib
matplotlib.use('Agg') if not IN_COLAB else None
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.spatial.transform import Rotation
from scipy.stats import spearmanr

def detect_gpu():
    info = {'torch': None, 'cuda_available': False, 'devices': []}
    try:
        import torch
        info['torch'] = torch.__version__
        info['cuda_available'] = bool(torch.cuda.is_available())
        if info['cuda_available']:
            for i in range(torch.cuda.device_count()):
                p = torch.cuda.get_device_properties(i)
                info['devices'].append({'name': p.name, 'memory_gb': round(p.total_memory / 2**30, 1)})
    except ImportError:
        try:
            out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                                 capture_output=True, text=True, timeout=10)
            if out.returncode == 0 and out.stdout.strip():
                info['devices'] = [{'name': l.strip()} for l in out.stdout.strip().splitlines()]
        except Exception:
            pass
    return info

def ram_gb():
    try:
        for line in open('/proc/meminfo'):
            if line.startswith('MemTotal'):
                return round(int(line.split()[1]) / 2**20, 1)
    except Exception:
        return None

ENV = {'platform': PLATFORM, 'python': sys.version.split()[0], 'os': platform.platform(),
       'cpu_count': os.cpu_count(), 'ram_gb': ram_gb(), 'gpu': detect_gpu(),
       'numpy': np.__version__, 'scipy': scipy.__version__, 'pandas': pd.__version__}
DEVICE = 'cuda' if ENV['gpu']['cuda_available'] else 'cpu'
print(json.dumps(ENV, indent=2))
print('Device:', DEVICE, '(the data, pose and metric sections run on CPU; ACE training needs a CUDA GPU)')

## 3. Configuration and reproducibility

*What:* one structured `CFG` dictionary, fixed seeds (0, 1, 2), directory layout, an append-only experiment log, a resume-safe cache helper and the `STATUS` registry. *Why:* every later number must be traceable to a configuration and a seed, and an interrupted run must resume without repeating finished work. *Expected output:* the saved configuration file and the directory tree.

**Licence gate.** The 7-Scenes data is non-commercial and must be accepted by the person running the notebook. Set `ACCEPT_7SCENES_LICENSE = True` only after reading https://www.microsoft.com/en-us/research/project/rgb-d-dataset-7-scenes/ .

In [ ]:
ACCEPT_7SCENES_LICENSE = True      # licence read and accepted by the person running this notebook (set False to disable all downloads)

ROOT = Path(os.environ.get('P22_WORK') or ('/content/p22' if IN_COLAB else '/kaggle/working/p22' if IN_KAGGLE else './p22_work')).resolve()
# Persistent storage: on Colab, caches, results and trained ACE heads live on Google Drive so a disconnected session can resume.
PERSIST_ROOT = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PERSIST_ROOT = Path('/content/drive/MyDrive/p22_persist'); PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print('Drive not mounted, results stay in the (temporary) session storage:', type(e).__name__)
if IN_KAGGLE:
    # Kaggle has no Drive: results live in /kaggle/working (kept as notebook output when the version is saved). To resume a later session,
    # add the earlier version's output as an input; its p22_persist folder is copied back here once, if this folder is still empty.
    PERSIST_ROOT = Path('/kaggle/working/p22_persist'); PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
    if not any(PERSIST_ROOT.iterdir()):
        for prev in sorted(Path('/kaggle/input').glob('*/p22_persist')) + sorted(Path('/kaggle/input').glob('*/*/p22_persist')):
            shutil.copytree(prev, PERSIST_ROOT, dirs_exist_ok=True); print('restored earlier results from', prev); break
_base = PERSIST_ROOT or ROOT
DIRS = {'data': ROOT / 'data', 'ext': ROOT / 'ext'}
DIRS.update({k: _base / k for k in ['cache', 'results/tables', 'results/figures', 'results/predictions', 'results/logs', 'ace_runs']})
DIRS['runs'] = DIRS['ace_runs']
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    'scenes': ['chess', 'stairs'],
    'gt_variants': ['dslam', 'sfm'],              # names of the pseudo-GT repo folders (pgt/dslam, pgt/sfm)
    'seeds': [0, 1, 2],
    'thresholds': {'trans_cm': 5.0, 'rot_deg': 5.0},   # usable = both within threshold
    'distance_bins': 5,                          # equal-count bins
    'usable_share': 0.5,                         # bin is usable if >= 50 % of its queries are usable
    'orientation_split_deg': 30.0,
    'bootstrap': {'block_len': 50, 'n_boot': 2000, 'ci': 0.95},
    'netvlad': {'model': 'VGG16-NetVLAD-Pitts30K', 'whiten': True, 'dim': 4096, 'batch': 16,
                'k_grid': [2, 3, 5, 8, 12, 20], 'ridge': 1e-6,
                'k_rule': 'k minimising the MEDIAN TRANSLATION ERROR on the development sequence (queries) against the development map; ties -> smaller k'},
    'ace': {'seeds_to_run': [0, 1, 2], 'smoke_gate_min_acc_pct': 90.0, 'parity_tol_t_cm': 0.1, 'parity_tol_r_deg': 0.05,
            'repro_tol_pp': {'chess': 1.0, 'stairs': 3.0}, 'buffer_default': 8000000, 'buffer_fallback': 4000000},
    'netvlad_sanity_factor': 3.0,
    'densities': {'fractions': [1.0, 0.5, 0.25, 0.10], 'uniform_stride_50pct': 2, 'random_thinning_seed': 20260101,
                  'seeds': {0.5: [0, 1, 2], 0.25: [0], 0.10: [0]}},      # ACE seeds per training density (GPU budget: 20 extra runs)
    'dev_sequence_rule': 'last training sequence in sorted order (fixed before any result existed)',
    'urls': {
        'ms_scene_zip': 'http://download.microsoft.com/download/2/8/5/28564B23-0828-408F-8631-23B1EFF1DAC8/{scene}.zip',
        'pgt_repo': 'https://github.com/tsattler/visloc_pseudo_gt_limitations.git',
        'ace_repo': 'https://github.com/nianticlabs/ace',
    },
    'image_size_hw': (480, 640),
    'rgb_focal_default': 525.0,
}

# Published reference values (from papers, NOT measured here). Never merged into measured tables.
PUBLISHED_REFERENCE = {
    'ace_sfm_acc_5cm5deg':   {'chess': 100.0, 'stairs': 81.9, 'source': 'ACE paper supplement (per-scene, SfM poses)'},
    'ace_dslam_acc_5cm5deg': {'chess': 96.7,  'stairs': 68.9, 'source': 'Bian et al. 2025, Table 4 (third-party run, approximate)'},
    'densevlad_top1_median': {'chess': (21.0, 12.5), 'stairs': (25.0, 15.8), 'source': 'Sattler et al. CVPR 2019, Table 2 (DenseVLAD, NOT NetVLAD)'},
}

def set_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
set_seeds(0)

LOG_PATH = DIRS['results/logs'] / 'experiment_log.jsonl'
def log_event(kind, **kw):
    rec = {'t': time.strftime('%Y-%m-%d %H:%M:%S'), 'kind': kind, **kw}
    with open(LOG_PATH, 'a') as f:
        f.write(json.dumps(rec, default=str) + '\n')

STATUS = {}      # name -> {'status': PASS|FAIL|NOT_RUN, 'detail': str}
def record(name, status, detail=''):
    assert status in ('PASS', 'FAIL', 'NOT_RUN')
    STATUS[name] = {'status': status, 'detail': detail}
    log_event('check', name=name, status=status, detail=detail)
    print(f'[{status:7s}] {name}' + (f'  -- {detail}' if detail else ''))

def cached_json(path, fn, force=False):
    """Resume-safe: reuse the file if it exists, otherwise compute and store."""
    path = Path(path)
    if path.exists() and not force:
        return json.loads(path.read_text())
    out = fn(); path.write_text(json.dumps(out, indent=1, default=str)); return out

def run_git(args, cwd=None):
    return subprocess.run(['git', *args], cwd=cwd, capture_output=True, text=True)

def clone_pinned(url, dest, depth=1):
    """Clone once; return the commit hash so the dependency version is recorded."""
    dest = Path(dest)
    if not dest.exists():
        r = run_git(['clone', '--depth', str(depth), url, str(dest)])
        if r.returncode != 0:
            raise RuntimeError(f'git clone failed for {url}: {r.stderr.strip()[:300]}')
    return run_git(['rev-parse', 'HEAD'], cwd=dest).stdout.strip()

CFG['environment'] = ENV
(DIRS['results/logs'] / 'config_M1.json').write_text(json.dumps(CFG, indent=1, default=str))
log_event('session_start', platform=PLATFORM, root=str(ROOT))
print('ROOT =', ROOT); print('PERSIST_ROOT =', PERSIST_ROOT); print({k: str(v) for k, v in DIRS.items()})

## 4. Dataset downloading and extraction

*What:* (a) fetch the external pose repository of Brachmann et al. (ICCV 2021), which holds **both** ground-truth variants (`pgt/dslam`, `pgt/sfm`) for every 7-Scenes frame together with the official train/test sequence split; (b) probe the Microsoft host and, only if the licence flag is set, download and restructure the scene images into the directory layout ACE expects. *Why:* the pose repository is small and lets all pose-level validation run anywhere; the images are needed for NetVLAD and ACE. *Expected output:* the pose repo commit hash; a reachability report for the image host; restructured scene folders when the download is permitted and reachable.

*Design note.* The restructuring below re-implements the logic of ACE's `datasets/setup_7scenes.py` (same folder layout, same RGB-sensor registration constants `d_to_rgb`) without its interactive prompts. Parity with ACE's own `read_pose_data` is tested in Section 7.

In [ ]:
PGT_COMMIT = clone_pinned(CFG['urls']['pgt_repo'], DIRS['ext'] / 'visloc_pseudo_gt_limitations')
PGT_DIR = DIRS['ext'] / 'visloc_pseudo_gt_limitations' / 'pgt'
CFG['provenance'] = {'pgt_repo_commit': PGT_COMMIT}
print('pose repository commit:', PGT_COMMIT)
for v in CFG['gt_variants']:
    for s in CFG['scenes']:
        for sp in ['train', 'test']:
            assert (PGT_DIR / v / '7scenes' / f'{s}_{sp}.txt').exists(), (v, s, sp)
record('pose_files_present', 'PASS', f'{len(CFG["gt_variants"])} variants x {len(CFG["scenes"])} scenes x 2 splits')

# --- image host reachability and download ---------------------------------------------------
D_TO_RGB = np.array([   # depth -> RGB sensor transform, identical to ACE setup_7scenes.py
    [ 9.9996518012567637e-01,  2.6765126468950343e-03, -7.9041012313000904e-03, -2.5558943178152542e-02],
    [-2.7409311281316700e-03,  9.9996302803027592e-01, -8.1504520778013286e-03,  1.0109636268061706e-04],
    [ 7.8819942130445332e-03,  8.1718328771890631e-03,  9.9993554558014031e-01,  2.0318321729487039e-03],
    [0, 0, 0, 1]])

def probe_image_host(scene='chess', timeout=20):
    url = CFG['urls']['ms_scene_zip'].format(scene=scene)
    try:
        req = urllib.request.Request(url, method='HEAD')
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return {'reachable': True, 'status': r.status, 'bytes': r.headers.get('Content-Length'), 'url': url}
    except Exception as e:
        return {'reachable': False, 'error': f'{type(e).__name__}: {e}', 'url': url}

# Optional: folder holding chess.zip / stairs.zip you downloaded yourself (e.g. a mounted Google Drive folder).
# If the files are there they are used and nothing is downloaded.
LOCAL_ZIP_DIR = Path('/content/drive/MyDrive/7scenes')      # folder containing chess.zip and stairs.zip (used if present, otherwise the notebook downloads)
if IN_COLAB and not LOCAL_ZIP_DIR.exists():
    try:
        from google.colab import drive; drive.mount('/content/drive')
    except Exception as e: print('Drive not mounted:', type(e).__name__)
if os.environ.get('P22_ZIPS'): LOCAL_ZIP_DIR = Path(os.environ['P22_ZIPS'])
if IN_KAGGLE:    # chess.zip / stairs.zip added as a Kaggle dataset are used directly (no 3 GB downloads)
    _hits = sorted(Path('/kaggle/input').rglob('chess.zip'))           # any folder depth, whatever the dataset is called
    if _hits: LOCAL_ZIP_DIR = _hits[0].parent; print('using zips from', LOCAL_ZIP_DIR)
JUNK_MEMBERS = ('thumbs.db', '.ds_store')       # Windows/macOS clutter; a bad CRC on these is harmless (the 7-Scenes archives contain such files)
EXTRACT_REPORT = {}

def safe_extract(zip_path, dest, tag):
    """Extract a zip member by member. A CRC/read error on a junk file is recorded and skipped; on ANY other
    member it raises, so a truncated or corrupt image/pose can never slip through."""
    skipped = []
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            if info.is_dir(): continue
            try:
                z.extract(info, dest)
            except (zipfile.BadZipFile, OSError, EOFError, RuntimeError) as e:
                if Path(info.filename).name.lower() in JUNK_MEMBERS:
                    skipped.append(info.filename); (Path(dest) / info.filename).unlink(missing_ok=True)
                else:
                    raise RuntimeError(f'corrupt member {info.filename!r} in {zip_path}: {e}')
    EXTRACT_REPORT[tag] = skipped
    return skipped

def download_scene(scene, dest_root):
    """Obtain <scene>.zip (local copy or download), extract it and its nested sequence zips. Resumable via a marker file."""
    dest_root = Path(dest_root); dest_root.mkdir(parents=True, exist_ok=True)
    scene_dir = dest_root / scene; marker = scene_dir / '.extract_complete'
    if marker.exists(): return scene_dir
    zpath = dest_root / f'{scene}.zip'
    if not scene_dir.exists() or not any(scene_dir.glob('*.zip')) and not any(scene_dir.glob('seq-*')):
        local = (LOCAL_ZIP_DIR / f'{scene}.zip') if LOCAL_ZIP_DIR else None
        if local is not None and local.exists():
            zpath = local; print(f'using local archive {local}')
        else:
            if not zpath.exists():
                part = zpath.with_suffix('.part')
                urllib.request.urlretrieve(CFG['urls']['ms_scene_zip'].format(scene=scene), part)
                expect = HOST_SIZES.get(scene)
                if expect and part.stat().st_size != int(expect):
                    raise IOError(f'incomplete download of {scene}.zip: {part.stat().st_size} of {expect} bytes')
                part.rename(zpath)
        safe_extract(zpath, dest_root, f'{scene}/outer')
        if zpath.parent == dest_root: zpath.unlink()
    for inner in sorted(scene_dir.glob('*.zip')):          # also finishes a previously interrupted run
        safe_extract(inner, scene_dir, f'{scene}/{inner.name}'); inner.unlink()
    marker.write_text('ok')
    return scene_dir

HOST_PROBE = probe_image_host()
HOST_SIZES = {'chess': HOST_PROBE.get('bytes')}
log_event('image_host_probe', **HOST_PROBE)
print(HOST_PROBE)
SOURCE_ROOT = DIRS['data'] / '7scenes_source'
REAL_IMAGES_AVAILABLE = False
if not ACCEPT_7SCENES_LICENSE:
    record('image_download', 'NOT_RUN', 'ACCEPT_7SCENES_LICENSE is False (licence must be accepted by the user)')
elif not HOST_PROBE['reachable']:
    record('image_download', 'NOT_RUN', 'Microsoft host unreachable from this environment: ' + HOST_PROBE['error'][:160])
else:
    try:
        for s in CFG['scenes']:
            download_scene(s, SOURCE_ROOT)
        REAL_IMAGES_AVAILABLE = all((SOURCE_ROOT / s / '.extract_complete').exists() for s in CFG['scenes'])
        print('junk members skipped:', {k: v for k, v in EXTRACT_REPORT.items() if v})
        record('image_download', 'PASS' if REAL_IMAGES_AVAILABLE else 'FAIL', f'scenes in {SOURCE_ROOT}')
    except Exception as e:
        record('image_download', 'FAIL', f'{type(e).__name__}: {e}')

## 5. Dataset exploration

*What:* inventory of the real pose files: frames per sequence, train/test sequence identities, trajectory extent, and the **step between consecutive frames** (translation in cm, rotation in degrees). *Why:* (i) the inventory feeds the integrity checks; (ii) the consecutive-frame step quantifies how near-duplicate neighbouring video frames are, which justifies block bootstrapping and uniform-stride thinning. *Expected output:* an inventory table, a step-size table and a trajectory figure. All numbers come from the real pose files.

In [ ]:
def read_pgt_poses(path):
    """Parse a pseudo-GT pose file. Line format: <seq/frame.color.png> qw qx qy qz tx ty tz [focal].
    The stored pose maps WORLD->CAMERA; this returns CAMERA->WORLD 4x4 matrices (what the metrics use)."""
    keys, quats, ts, focals = [], [], [], []
    for line in open(path):
        p = line.split()
        if not p: continue
        keys.append(p[0].replace('.color.png', '').replace('.png', ''))
        quats.append([float(p[2]), float(p[3]), float(p[4]), float(p[1])])        # scipy wants x y z w
        ts.append([float(p[5]), float(p[6]), float(p[7])])
        focals.append(float(p[8]) if len(p) > 8 else np.nan)
    R_w2c = Rotation.from_quat(np.array(quats)).as_matrix()
    t_w2c = np.array(ts)
    T = np.tile(np.eye(4), (len(keys), 1, 1))
    T[:, :3, :3] = np.transpose(R_w2c, (0, 2, 1))                                  # R_c2w = R_w2c^T
    T[:, :3, 3] = -np.einsum('nij,nj->ni', np.transpose(R_w2c, (0, 2, 1)), t_w2c)  # c = -R_w2c^T t_w2c
    return np.array(keys), T, np.array(focals)

class PoseSet:
    """Keys + camera-to-world poses (+focal) with a fixed row order. Scoring always aligns by KEY, never by position."""
    def __init__(self, keys, T, focal=None, name=''):
        self.keys = np.asarray(keys); self.T = np.asarray(T, dtype=np.float64)
        self.focal = np.full(len(self.keys), np.nan) if focal is None else np.asarray(focal)
        self.name = name
        assert self.T.shape == (len(self.keys), 4, 4)
        assert len(set(self.keys)) == len(self.keys), f'duplicate keys in {name}'
    def __len__(self): return len(self.keys)
    @property
    def centres(self): return self.T[:, :3, 3]
    @property
    def R(self): return self.T[:, :3, :3]
    @property
    def seq(self): return np.array([k.split('/')[0] for k in self.keys])
    @property
    def frame(self): return np.array([int(k.split('frame-')[1]) for k in self.keys])
    def subset(self, mask, name=None):
        mask = np.asarray(mask); return PoseSet(self.keys[mask], self.T[mask], self.focal[mask], name or self.name)
    def by_key(self, keys):
        idx = {k: i for i, k in enumerate(self.keys)}
        return PoseSet(keys, self.T[[idx[k] for k in keys]], self.focal[[idx[k] for k in keys]], self.name)

def load_poseset(scene, variant, split):
    k, T, f = read_pgt_poses(PGT_DIR / variant / '7scenes' / f'{scene}_{split}.txt')
    return PoseSet(k, T, f, name=f'{scene}/{variant}/{split}')

POSES = {(s, v, sp): load_poseset(s, v, sp) for s in CFG['scenes'] for v in CFG['gt_variants'] for sp in ['train', 'test']}

rows = []
for (s, v, sp), ps in POSES.items():
    if v != 'sfm': continue
    for q in sorted(set(ps.seq)):
        m = ps.seq == q
        rows.append({'scene': s, 'split': sp, 'sequence': q, 'frames': int(m.sum()),
                     'first_frame': int(ps.frame[m].min()), 'last_frame': int(ps.frame[m].max())})
INVENTORY = pd.DataFrame(rows)
INVENTORY.to_csv(DIRS['results/tables'] / 'dataset_inventory.csv', index=False)
print(INVENTORY.to_string(index=False))
print('\nframes per (scene, split):'); print(INVENTORY.groupby(['scene', 'split'])['frames'].sum().to_string())

In [ ]:
# consecutive-frame steps inside each sequence (sorted by frame index): how near-duplicate are neighbours?
def consecutive_steps(ps):
    t, r = [], []
    for q in sorted(set(ps.seq)):
        idx = np.where(ps.seq == q)[0]; idx = idx[np.argsort(ps.frame[idx])]
        c = ps.centres[idx]; R = ps.R[idx]
        t.append(np.linalg.norm(np.diff(c, axis=0), axis=1) * 100)
        Rrel = np.einsum('nji,njk->nik', R[:-1], R[1:])
        r.append(np.degrees(np.arccos(np.clip((np.trace(Rrel, axis1=1, axis2=2) - 1) / 2, -1, 1))))
    return np.concatenate(t), np.concatenate(r)

rows = []
for s in CFG['scenes']:
    for v in CFG['gt_variants']:
        ps = POSES[(s, v, 'test')]; t, r = consecutive_steps(ps)
        ext = ps.centres.max(0) - ps.centres.min(0)
        rows.append({'scene': s, 'gt': v, 'median_step_cm': np.median(t), 'p95_step_cm': np.percentile(t, 95),
                     'median_step_deg': np.median(r), 'p95_step_deg': np.percentile(r, 95),
                     'extent_xyz_m': np.round(ext, 2).tolist()})
STEPS = pd.DataFrame(rows); STEPS.to_csv(DIRS['results/tables'] / 'consecutive_frame_steps.csv', index=False)
print(STEPS.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, s in zip(axes, CFG['scenes']):
    tr, te = POSES[(s, 'sfm', 'train')], POSES[(s, 'sfm', 'test')]
    allc = np.vstack([tr.centres, te.centres]); mu = allc.mean(0)
    _, _, Vt = np.linalg.svd(allc - mu, full_matrices=False)          # project onto the two main axes of the trajectory
    P = lambda c: (c - mu) @ Vt[:2].T
    ax.scatter(*P(tr.centres).T, s=2, c='tab:blue', label=f'train ({len(tr)})')
    ax.scatter(*P(te.centres).T, s=2, c='tab:red', label=f'test ({len(te)})')
    ax.set_title(f'{s}: camera centres (SfM poses)'); ax.set_xlabel('principal axis 1 (m)'); ax.set_ylabel('principal axis 2 (m)')
    ax.set_aspect('equal'); ax.legend(markerscale=4)
plt.tight_layout(); plt.savefig(DIRS['results/figures'] / 'trajectories.png', dpi=200); plt.show()

## 6. Image and pose loading

*What:* the frame table (key ↔ pose ↔ image path), the scene scanner for real images (dimensions, unreadable files) and the restructuring into ACE's folder layout. Image–pose correspondence is established **by key** (`seq-XX/frame-XXXXXX`), never by independently sorted file lists. *Expected output:* with real images, a scan report; without them, an explicit `NOT_RUN`. The loader logic itself is unit-tested in Section 10 on a small *synthetic fixture* that mimics the Microsoft layout (labelled as such there).

In [ ]:
def seq_dirname_from_split_line(line):
    """TrainSplit.txt/TestSplit.txt lines look like 'sequence3' -> 'seq-03' (same rule as ACE's setup script)."""
    return 'seq-' + line.strip()[8:].zfill(2)

def read_rgb_pose_original(path):
    return np.loadtxt(path)

def orthonormalise_rotation(R):
    """Nearest proper rotation (SVD projection). The original 7-Scenes pose.txt files store rotations with few decimals, so their
    rotation blocks deviate from orthonormal by ~3e-4 (measured in Section 8). Near the identity the trace formula
    arccos((tr-1)/2) amplifies such a deviation to ~1 degree, so unprojected matrices must not be scored with it."""
    U, _, Vt = np.linalg.svd(R); Rn = U @ Vt
    if np.linalg.det(Rn) < 0: U[:, -1] *= -1; Rn = U @ Vt
    return Rn

def calibrate_dslam_pose(T_depth_sensor, project=True):
    """Original 7-Scenes poses refer to the depth sensor; ACE's 'calibrated' variant maps them to the RGB sensor.
    project=True additionally projects the rotation block to the nearest proper rotation (see orthonormalise_rotation)."""
    T = T_depth_sensor @ np.linalg.inv(D_TO_RGB)
    if project: T = T.copy(); T[:3, :3] = orthonormalise_rotation(T[:3, :3])
    return T

def write_pose_txt(path, T):
    with open(path, 'w') as f:
        for row in T: f.write(' '.join(repr(float(x)) for x in row) + '\n')

def restructure_scene(scene, source_root, out_root, variant_tag, pgt_poseset_by_split=None, focal=None, link_images=True):
    """Create  <out_root>/<scene>_<variant_tag>/{train,test}/{rgb,poses,calibration}  (ACE layout).
    variant_tag 'dslam_cal': poses = original depth-sensor poses registered to RGB (calibrate_dslam_pose).
    variant_tag 'sfm_pgt'  : poses = pseudo-GT poses from the pose repository (pgt_poseset_by_split)."""
    src = Path(source_root) / scene
    tgt = Path(out_root) / f'{scene}_{variant_tag}'
    for split, split_file in [('train', 'TrainSplit.txt'), ('test', 'TestSplit.txt')]:
        for sub in ['rgb', 'poses', 'calibration']:
            (tgt / split / sub).mkdir(parents=True, exist_ok=True)
        seqs = [seq_dirname_from_split_line(l) for l in open(src / split_file) if l.strip()]
        for seq in seqs:
            for img in sorted(p for p in (src / seq).iterdir() if p.name.endswith('color.png')):
                base = img.name[:-len('.color.png')]
                dst = tgt / split / 'rgb' / f'{seq}-{img.name}'
                if not dst.exists():
                    try:
                        if not link_images: raise OSError
                        os.symlink(img.resolve(), dst)
                    except OSError:
                        shutil.copy(img, dst)
                pose_dst = tgt / split / 'poses' / f'{seq}-{base}.pose.txt'
                if variant_tag == 'dslam_cal':
                    write_pose_txt(pose_dst, calibrate_dslam_pose(read_rgb_pose_original(src / seq / f'{base}.pose.txt')))
                    f_val = focal or CFG['rgb_focal_default']
                else:
                    ps = pgt_poseset_by_split[split]; i = list(ps.keys).index(f'{seq}/{base}')
                    write_pose_txt(pose_dst, ps.T[i]); f_val = ps.focal[i]
                (tgt / split / 'calibration' / f'{seq}-{base}.calibration.txt').write_text(str(f_val))
    return tgt

def verify_image_pose_parity(scene_dir, expect_hw=None, expect_keys=None):
    """Hard checks that images, poses and calibration files correspond one-to-one by key. Raises AssertionError."""
    report = {}
    for split in ['train', 'test']:
        d = Path(scene_dir) / split
        rgb = {p.name[:-len('.color.png')] for p in (d / 'rgb').iterdir() if p.name.endswith('.color.png')}
        pos = {p.name[:-len('.pose.txt')] for p in (d / 'poses').iterdir() if p.name.endswith('.pose.txt')}
        cal = {p.name[:-len('.calibration.txt')] for p in (d / 'calibration').iterdir() if p.name.endswith('.calibration.txt')}
        assert rgb == pos == cal, f'{split}: image/pose/calibration sets differ: only-img={sorted(rgb-pos)[:3]} only-pose={sorted(pos-rgb)[:3]} only-cal={sorted(cal-rgb)[:3]}'
        bad, dims = [], set()
        from PIL import Image
        for name in sorted(rgb):
            try:
                with Image.open(d / 'rgb' / f'{name}.color.png') as im:
                    im.verify()
                with Image.open(d / 'rgb' / f'{name}.color.png') as im:
                    dims.add(im.size)
            except Exception as e:
                bad.append((name, str(e)[:60]))
            T = np.loadtxt(d / 'poses' / f'{name}.pose.txt')
            assert T.shape == (4, 4) and np.isfinite(T).all(), f'bad pose file {name}'
        assert not bad, f'unreadable images: {bad[:3]}'
        if expect_hw: assert dims == {(expect_hw[1], expect_hw[0])}, f'{split}: image sizes {dims}'
        if expect_keys is not None:
            want = {k.replace('/', '-') for k in expect_keys[split]}
            assert rgb == want, f'{split}: frame set differs from the pose repository ({len(rgb ^ want)} frames)'
        report[split] = {'frames': len(rgb), 'sizes': sorted(dims)}
    return report

if REAL_IMAGES_AVAILABLE:
    SCAN = {}
    for s in CFG['scenes']:
        SCAN[s] = {}
        for tag, kw in [('dslam_cal', {}), ('sfm_pgt', {'pgt_poseset_by_split': {sp: POSES[(s, 'sfm', sp)] for sp in ['train', 'test']}})]:
            tgt = restructure_scene(s, SOURCE_ROOT, DIRS['data'], tag, **kw)
            exp = {sp: list(POSES[(s, 'sfm', sp)].keys) for sp in ['train', 'test']}
            SCAN[s][tag] = verify_image_pose_parity(tgt, expect_hw=CFG['image_size_hw'], expect_keys=exp)
    print(json.dumps(SCAN, indent=1))
    record('real_images_loaded_and_matched', 'PASS', 'all frames matched by key, sizes 640x480, none unreadable')
else:
    record('real_images_loaded_and_matched', 'NOT_RUN', 'real images unavailable in this environment (see image_download)')

### 6 (continued). Train / development / test data structures

*What:* for every scene and ground-truth variant, `train` (all official training sequences), `dev` (one held-out training sequence, chosen by the pre-registered rule: the last training sequence), `dev_map` (training minus dev) and `test` (official test sequences). *Why:* all tuning (k for top-k retrieval, bootstrap block length) happens on `dev` against `dev_map`; `test` is never used for tuning. *Expected output:* a size table and passing leakage assertions.

In [ ]:
SPLITS = {}
rows = []
for s in CFG['scenes']:
    for v in CFG['gt_variants']:
        tr, te = POSES[(s, v, 'train')], POSES[(s, v, 'test')]
        dev_seq = sorted(set(tr.seq))[-1]                      # pre-registered rule
        dev = tr.subset(tr.seq == dev_seq, f'{s}/{v}/dev')
        dev_map = tr.subset(tr.seq != dev_seq, f'{s}/{v}/dev_map')
        SPLITS[(s, v)] = {'train': tr, 'dev': dev, 'dev_map': dev_map, 'test': te, 'dev_seq': dev_seq}
        assert set(tr.seq).isdisjoint(set(te.seq)), 'train and test share a sequence'
        assert set(tr.keys).isdisjoint(set(te.keys)), 'train and test share a frame'
        assert set(dev.keys).isdisjoint(set(dev_map.keys)) and set(dev.keys) | set(dev_map.keys) == set(tr.keys)
        assert set(dev.keys).isdisjoint(set(te.keys))
        rows.append({'scene': s, 'gt': v, 'train_seqs': sorted(set(tr.seq)), 'dev_seq': dev_seq,
                     'test_seqs': sorted(set(te.seq)), 'n_train': len(tr), 'n_dev': len(dev), 'n_dev_map': len(dev_map), 'n_test': len(te)})
SPLIT_ROWS = rows
print(pd.DataFrame(SPLIT_ROWS).to_string(index=False))

def thin_uniform(ps, stride):
    """Keep every `stride`-th frame (in frame order) inside each sequence. stride=2 gives the 50 % density."""
    keep = np.zeros(len(ps), dtype=bool)
    for q in sorted(set(ps.seq)):
        idx = np.where(ps.seq == q)[0]; idx = idx[np.argsort(ps.frame[idx])]
        keep[idx[::stride]] = True
    return ps.subset(keep, ps.name + f'/stride{stride}')
_t = thin_uniform(SPLITS[('chess', 'sfm')]['train'], 2)
assert len(_t) == len(SPLITS[('chess', 'sfm')]['train']) // 2 and set(_t.keys) <= set(SPLITS[('chess', 'sfm')]['train'].keys)
record('split_structures_no_leakage', 'PASS', 'sequence- and frame-level disjointness asserted for 2 scenes x 2 variants; uniform thinning keeps every 2nd frame')

## 7. Pose transformation utilities

*What:* pose inversion, camera-centre extraction and validity checks, then **four independent validations of the coordinate conventions**: (1) rotations in every real pose file are orthonormal with determinant +1; (2) our parser agrees with a direct `numpy.linalg.inv` computation; (3) our parser agrees with **ACE's own `read_pose_data`**; (4) our `D_TO_RGB` constant equals the one in ACE's `setup_7scenes.py`. *Why:* a convention error (world→camera versus camera→world) would silently corrupt every translation error. *Expected output:* all checks pass, plus the measured gap between the two ground-truth variants.

In [ ]:
def invert_pose(T):
    """Rigid inverse using R^T (not a general matrix inverse)."""
    T = np.asarray(T); out = np.tile(np.eye(4), T.shape[:-2] + (1, 1))
    Rt = np.swapaxes(T[..., :3, :3], -1, -2)
    out[..., :3, :3] = Rt
    out[..., :3, 3] = -np.einsum('...ij,...j->...i', Rt, T[..., :3, 3])
    return out

def camera_centre_from_w2c(T_w2c):
    """c = -R^T t for a world->camera pose."""
    return -np.einsum('...ji,...j->...i', T_w2c[..., :3, :3], T_w2c[..., :3, 3])

def rotation_validity(R):
    """Max deviation from orthonormality and from det=+1."""
    R = np.asarray(R); I = np.eye(3)
    ortho = np.abs(np.einsum('nji,njk->nik', R, R) - I).max()
    det = np.abs(np.linalg.det(R) - 1).max()
    return float(ortho), float(det)

# (1) validity of every real pose
worst_o, worst_d = 0, 0
for key, ps in POSES.items():
    o, d_ = rotation_validity(ps.R); worst_o, worst_d = max(worst_o, o), max(worst_d, d_)
    assert np.allclose(ps.T[:, 3, :], [0, 0, 0, 1]) and np.isfinite(ps.T).all(), key
print(f'max |R^T R - I| = {worst_o:.2e}, max |det R - 1| = {worst_d:.2e} over {sum(len(p) for p in POSES.values())} real poses')

# (2) independent re-computation with a general matrix inverse on 300 random lines per file
rng = np.random.default_rng(0); worst_inv = 0
for (s, v, sp), ps in POSES.items():
    lines = [l.split() for l in open(PGT_DIR / v / '7scenes' / f'{s}_{sp}.txt') if l.strip()]
    for i in rng.choice(len(lines), 300, replace=False):
        p = lines[i]
        q = [float(p[2]), float(p[3]), float(p[4]), float(p[1])]
        W2C = np.eye(4); W2C[:3, :3] = Rotation.from_quat(q).as_matrix(); W2C[:3, 3] = [float(p[5]), float(p[6]), float(p[7])]
        j = list(ps.keys).index(p[0].replace('.color.png', ''))
        worst_inv = max(worst_inv, np.abs(np.linalg.inv(W2C) - ps.T[j]).max())
        worst_inv = max(worst_inv, np.abs(camera_centre_from_w2c(W2C) - ps.centres[j]).max())
print(f'max |inv(w2c) - c2w| (and centre formula) = {worst_inv:.2e}')
ok_conv = worst_o < 1e-5 and worst_d < 1e-5 and worst_inv < 1e-9
record('pose_conventions', 'PASS' if ok_conv else 'FAIL', f'orthonormality {worst_o:.1e}, det {worst_d:.1e}, inverse/centre {worst_inv:.1e}')

In [ ]:
# (3) parity with ACE's own pose reader and (4) parity of the depth->RGB constant (read from ACE's source, not retyped)
import ast
ACE_DIR = DIRS['ext'] / 'ace'
try:
    ACE_COMMIT = clone_pinned(CFG['urls']['ace_repo'], ACE_DIR)
    CFG['provenance']['ace_repo_commit'] = ACE_COMMIT
    spec = importlib.util.spec_from_file_location('ace_dataset_util', ACE_DIR / 'datasets' / 'dataset_util.py')
    ace_util = importlib.util.module_from_spec(spec); spec.loader.exec_module(ace_util)
    worst = 0
    for (s, v, sp), ps in POSES.items():
        ref = ace_util.read_pose_data(str(PGT_DIR / v / '7scenes' / f'{s}_{sp}.txt'))
        assert set(ref) == set(ps.keys), 'key sets differ from ACE parser'
        for k, T in zip(ps.keys, ps.T):
            worst = max(worst, np.abs(ref[k][0] - T).max())
    record('ace_read_pose_parity', 'PASS' if worst < 1e-9 else 'FAIL', f'max abs diff vs ACE read_pose_data = {worst:.1e} (ACE commit {ACE_COMMIT[:8]})')

    tree = ast.parse((ACE_DIR / 'datasets' / 'setup_7scenes.py').read_text())
    ace_d2rgb = None
    for node in tree.body:
        if isinstance(node, ast.Assign) and getattr(node.targets[0], 'id', '') == 'd_to_rgb':
            ace_d2rgb = np.array(ast.literal_eval(node.value.args[0]), dtype=float)
    assert ace_d2rgb is not None
    diff = np.abs(ace_d2rgb - D_TO_RGB).max()
    record('d_to_rgb_parity', 'PASS' if diff == 0 else 'FAIL', f'max abs diff vs ACE setup_7scenes.py constant = {diff:.1e}')
except Exception as e:
    record('ace_read_pose_parity', 'NOT_RUN', f'ACE repository unavailable: {type(e).__name__}: {str(e)[:150]}')
    record('d_to_rgb_parity', 'NOT_RUN', 'ACE repository unavailable')

## 8. Metric implementation

*What:* translation error (cm) between camera centres, rotation error (°) from the trace formula with clamping, accuracy within thresholds, key-aligned scoring, equal-count distance bins and the usable-range rule. *Why:* the metrics were specified up front: e_t = ‖t̂ − t‖₂ and e_R = arccos(clamp((tr(R̂ᵀR) − 1)/2, −1, 1)); translation is the camera **centre** (the c2w translation), not the w2c translation. Translation and rotation are never merged. *Expected output:* a self-test against an independent implementation (SciPy `Rotation.magnitude`) and unit tests of the bin and usable-range logic.

In [ ]:
def translation_error_cm(t_est, t_gt):
    """Euclidean distance between camera centres in centimetres (poses are in metres)."""
    return np.linalg.norm(np.asarray(t_est, float) - np.asarray(t_gt, float), axis=-1) * 100.0

def rotation_error_deg(R_est, R_gt):
    """e_R = arccos(clamp((trace(R_est^T R_gt) - 1)/2, -1, 1)) in degrees; accepts one matrix or a stack."""
    R_est = np.asarray(R_est, float); R_gt = np.asarray(R_gt, float)
    if R_est.ndim == 2: R_est, R_gt = R_est[None], R_gt[None]
    M = np.einsum('nji,njk->nik', R_est, R_gt)
    return np.degrees(np.arccos(np.clip((np.trace(M, axis1=1, axis2=2) - 1) / 2, -1.0, 1.0)))

def score_predictions(pred, gt):
    """Per-frame errors. Rows are matched by KEY; a missing or extra key raises (no silent dropping of hard frames)."""
    assert set(pred.keys) == set(gt.keys), f'prediction/GT key mismatch: {len(set(pred.keys) ^ set(gt.keys))} keys differ'
    p = pred.by_key(gt.keys)
    return pd.DataFrame({'key': gt.keys,
                         't_err_cm': translation_error_cm(p.centres, gt.centres),
                         'r_err_deg': rotation_error_deg(p.R, gt.R)})

def summarise_errors(df):
    th = CFG['thresholds']
    usable = (df.t_err_cm <= th['trans_cm']) & (df.r_err_deg <= th['rot_deg'])
    return {'n': len(df), 'median_t_cm': float(df.t_err_cm.median()), 'median_r_deg': float(df.r_err_deg.median()),
            'acc_5cm_5deg_pct': float(100 * usable.mean())}

def assign_equal_count_bins(d, k):
    """Bin index per query so that bin sizes differ by at most one (equal-count bins on the distance)."""
    order = np.argsort(d, kind='stable'); b = np.empty(len(d), dtype=int)
    for i, idx in enumerate(np.array_split(order, k)): b[idx] = i
    return b

def usable_range(d, usable, k=None, share=None):
    """Largest-distance bin whose usable share is >= `share`. If every bin qualifies the range is CENSORED (not reached)."""
    k = k or CFG['distance_bins']; share = share if share is not None else CFG['usable_share']
    d = np.asarray(d); usable = np.asarray(usable, bool); b = assign_equal_count_bins(d, k)
    shares = np.array([usable[b == i].mean() for i in range(k)])
    ok = shares >= share
    if ok.all(): return {'status': 'censored', 'shares': shares.tolist(), 'upper_edge': None}
    if not ok.any(): return {'status': 'none', 'shares': shares.tolist(), 'upper_edge': None}
    last = int(np.where(ok)[0].max())
    return {'status': 'reached', 'bin': last, 'shares': shares.tolist(), 'upper_edge': float(d[b == last].max())}

# --- self-test 1: our rotation error vs SciPy's independent implementation on 2000 random pairs -------------------
rng = np.random.default_rng(1)
A = Rotation.random(2000, random_state=1); B = Rotation.random(2000, random_state=2)
mine = rotation_error_deg(A.as_matrix(), B.as_matrix()); ref = np.degrees((A.inv() * B).magnitude())
d_scipy = np.abs(mine - ref).max()
# edge cases: identity, 180 degrees, numerically over-unity trace
I = np.eye(3); R180 = Rotation.from_euler('z', 180, degrees=True).as_matrix()
edge_ok = abs(rotation_error_deg(I, I)[0]) < 1e-6 and abs(rotation_error_deg(I, R180)[0] - 180) < 1e-4 \
          and np.isfinite(rotation_error_deg(I * (1 + 1e-9), I)).all()
record('metrics_vs_scipy', 'PASS' if d_scipy < 1e-4 and edge_ok else 'FAIL', f'max |ours - scipy| = {d_scipy:.1e} deg over 2000 pairs; edge cases ok={edge_ok}')

# --- self-test 2: bin and usable-range logic on constructed data (validity test for the rule itself) --------------
d_syn = np.linspace(1, 100, 100); k = 5
assert np.bincount(assign_equal_count_bins(d_syn, k)).tolist() == [20] * 5
assert np.bincount(assign_equal_count_bins(np.arange(103), 5)).max() - np.bincount(assign_equal_count_bins(np.arange(103), 5)).min() <= 1
r1 = usable_range(d_syn, d_syn <= 40, k, 0.5)       # first two bins fully usable, third bin (41-60) not
r2 = usable_range(d_syn, np.ones(100, bool), k, 0.5)
r3 = usable_range(d_syn, np.zeros(100, bool), k, 0.5)
ok_ur = r1['status'] == 'reached' and r1['bin'] == 1 and r1['upper_edge'] == 40.0 and r2['status'] == 'censored' and r3['status'] == 'none'
record('usable_range_logic', 'PASS' if ok_ur else 'FAIL', f"reached/censored/none cases handled: {r1['status']}, {r2['status']}, {r3['status']}")

### 8 (continued). Calibration cross-check and D-SLAM diagnostic (needs the real images and the metrics defined above)

*Calibration cross-check.* ACE's "calibrated" D-SLAM variant is `T_original @ inv(D_TO_RGB)`. The pose repository's `pgt/dslam` files should agree with it. This is the one check that can show whether "D-SLAM" is unambiguous; it is executed only when the original Microsoft pose files are present.

In [ ]:
def crosscheck_dslam_calibration(scene):
    """Compare calibrate_dslam_pose(original pose.txt) with pgt/dslam poses for every frame. Needs real downloads.
    Reports, besides the errors, (i) how far the rotation blocks are from orthonormal and (ii) the gap between the trace-formula
    metric and SciPy's metric. A gap > 1e-3 deg means a rotation is NOT a proper rotation (or a function was redefined);
    it is reported as such and is never hidden by loosening a tolerance."""
    rows = []
    for sp in ['train', 'test']:
        ps = POSES[(scene, 'dslam', sp)]
        To = np.stack([read_rgb_pose_original(SOURCE_ROOT / scene / (k + '.pose.txt')) for k in ps.keys])
        raw = np.stack([calibrate_dslam_pose(T, project=False) for T in To])      # as ACE writes them (not projected)
        mine = np.stack([calibrate_dslam_pose(T) for T in To])                      # rotation projected to SO(3)
        te = translation_error_cm(mine[:, :3, 3], ps.centres); re_ = rotation_error_deg(mine[:, :3, :3], ps.R)
        re_sp = np.degrees((Rotation.from_matrix(ps.R).inv() * Rotation.from_matrix(mine[:, :3, :3])).magnitude())   # SciPy projects to SO(3)
        dev = lambda R: float(np.abs(np.einsum('nji,njk->nik', R, R) - np.eye(3)).max())
        rows.append({'scene': scene, 'split': sp, 'median_t_cm': float(np.median(te)), 'max_t_cm': float(te.max()),
                     'median_r_deg': float(np.median(re_)), 'max_r_deg': float(re_.max()),
                     'median_r_deg_scipy': float(np.median(re_sp)), 'max_r_deg_scipy': float(re_sp.max()),
                     'metric_gap_max_deg': float(np.abs(re_ - re_sp).max()),
                     'orthodev_original': dev(To[:, :3, :3]), 'orthodev_raw_calibrated': dev(raw[:, :3, :3]), 'orthodev_projected': dev(mine[:, :3, :3]), 'orthodev_pgt': dev(ps.R),
                     'median_r_deg_raw_scipy': float(np.median(np.degrees((Rotation.from_matrix(ps.R).inv() * Rotation.from_matrix(raw[:, :3, :3])).magnitude())))})
    return rows

if REAL_IMAGES_AVAILABLE:
    cc = [r for s in CFG['scenes'] for r in crosscheck_dslam_calibration(s)]
    CC_DF = pd.DataFrame(cc); CC_DF.to_csv(DIRS['results/tables'] / 'dslam_calibration_crosscheck.csv', index=False)
    with pd.option_context('display.width', 250, 'display.max_columns', 20): print(CC_DF.round(5).to_string(index=False))
    gap_ok = all(r['metric_gap_max_deg'] < 1e-3 for r in cc)
    agree = all(r['max_t_cm'] < 1.0 and r['max_r_deg'] < 0.5 for r in cc)
    if gap_ok and agree: record('dslam_calibration_crosscheck', 'PASS', 'calibrated original poses agree with pgt/dslam (< 1 cm, < 0.5 deg)')
    elif not gap_ok: record('dslam_calibration_crosscheck', 'FAIL', 'two rotation-error computations disagree even after projection to SO(3); investigate, tolerance NOT loosened')
    else: record('dslam_calibration_crosscheck', 'FAIL', 'calibrated original poses and pgt/dslam disagree beyond 1 cm / 0.5 deg')
else:
    record('dslam_calibration_crosscheck', 'NOT_RUN', 'original Microsoft pose files unavailable')

*Diagnostic (executed only with real images; it does not change any pass/fail rule).* If the cross-check above disagrees, the next cell asks **which transformation of the original pose file reproduces the pose repository's D-SLAM poses**, and whether the residual is a constant offset (a calibration difference) or noise. Nothing is tuned: the candidates are fixed in advance and every one is reported.

In [ ]:
def dslam_diagnostic(scene):
    out = []
    Dinv = np.linalg.inv(D_TO_RGB)
    cands = {'To @ inv(D)  [ACE calibrated; the rule used above]': lambda T: T @ Dinv,
             'To (uncalibrated)':                                  lambda T: T,
             'To @ D':                                             lambda T: T @ D_TO_RGB,
             'inv(D) @ To':                                        lambda T: Dinv @ T,
             'D @ To':                                             lambda T: D_TO_RGB @ T}
    for sp in ['train', 'test']:
        ps = POSES[(scene, 'dslam', sp)]
        To = np.stack([read_rgb_pose_original(SOURCE_ROOT / scene / (k + '.pose.txt')) for k in ps.keys])
        for name, f in cands.items():
            M = np.stack([f(T) for T in To])
            out.append({'scene': scene, 'split': sp, 'candidate': name,
                        'median_t_cm': float(np.median(translation_error_cm(M[:, :3, 3], ps.centres))),
                        'median_r_deg': float(np.median(np.degrees((Rotation.from_matrix(ps.R).inv() * Rotation.from_matrix(M[:, :3, :3])).magnitude())))})
        # residual rotation of the ACE-calibrated candidate, expressed in the camera frame: constant (systematic) or scattered?
        M = np.stack([T @ Dinv for T in To])
        rv = Rotation.from_matrix(np.einsum('nji,njk->nik', ps.R, M[:, :3, :3])).as_rotvec()      # R_pgt^T R_cal
        out.append({'scene': scene, 'split': sp, 'candidate': 'residual rotvec mean (deg) / std (deg)',
                    'median_t_cm': np.nan, 'median_r_deg': np.nan,
                    'detail': f'mean={np.round(np.degrees(rv.mean(0)), 3).tolist()} std={np.round(np.degrees(rv.std(0)), 3).tolist()}'})
    return out
if REAL_IMAGES_AVAILABLE:
    DIAG_DF = pd.DataFrame([r for sc in CFG['scenes'] for r in dslam_diagnostic(sc)])
    DIAG_DF.to_csv(DIRS['results/tables'] / 'dslam_variant_diagnostic.csv', index=False)
    with pd.option_context('display.max_colwidth', 80, 'display.width', 250):
        print(DIAG_DF.round(4).to_string(index=False))
    exact = DIAG_DF.dropna(subset=['median_t_cm']).query('median_t_cm < 0.1 and median_r_deg < 0.05').groupby('candidate').size()
    print('\ncandidates matching pgt/dslam to < 0.1 cm and < 0.05 deg on every scene and split:', exact[exact == 2 * len(CFG['scenes'])].index.tolist() or 'NONE')

## 9. Distance-to-nearest-view implementation

*What:* d(q) = min_i ‖c_q − c_i‖₂ over the training camera centres, the index of that nearest view, and the relative orientation angle between the query and that view. *Why:* this is the study's explanatory variable. It is an **offline evaluation quantity**: neither method may receive it. The code enforces this by keeping the localiser interface image-only. *Expected output:* a hand-computed example, a brute-force cross-check on real poses and the interface guard.

In [ ]:
def nearest_view_distance(train, query):
    """OFFLINE EVALUATION ONLY. Returns d in cm, index of the nearest training view and orientation difference in degrees."""
    d, idx = cKDTree(train.centres).query(query.centres)
    return {'d_cm': d * 100.0, 'nn_index': idx, 'orient_deg': rotation_error_deg(query.R, train.R[idx])}

OFFLINE_EVAL_FUNCTIONS = ['nearest_view_distance', 'score_predictions', 'usable_range']

class Localiser:
    """Interface every method must implement. It sees ONLY the image: no pose, no distance, no sequence id."""
    def localise(self, image): raise NotImplementedError
assert list(inspect.signature(Localiser.localise).parameters) == ['self', 'image']
record('localiser_interface_image_only', 'PASS', 'Localiser.localise(self, image) is the only entry point; distance code lives in OFFLINE_EVAL_FUNCTIONS')

# --- hand-computed example --------------------------------------------------------------------------------------
def _ps(centres, yaw_deg=None):
    T = np.tile(np.eye(4), (len(centres), 1, 1)); T[:, :3, 3] = centres
    if yaw_deg is not None: T[:, :3, :3] = Rotation.from_rotvec(np.stack([np.zeros(len(centres)), np.zeros(len(centres)), np.radians(np.asarray(yaw_deg, float))], 1)).as_matrix()
    return PoseSet([f'seq-01/frame-{i:06d}' for i in range(len(centres))], T)
train_h = _ps([[0, 0, 0], [1, 0, 0], [0, 2, 0]], yaw_deg=[0, 0, 0])
query_h = _ps([[0.4, 0, 0], [1, 0.3, 0.4]], yaw_deg=[10, 40])
res = nearest_view_distance(train_h, query_h)
# query 0: nearest is train 0 at 0.40 m = 40 cm, orientation 10 deg; query 1: train 1 at sqrt(0.3^2+0.4^2) = 0.5 m = 50 cm, orientation 40 deg
hand_ok = np.allclose(res['d_cm'], [40.0, 50.0]) and res['nn_index'].tolist() == [0, 1] and np.allclose(res['orient_deg'], [10.0, 40.0], atol=1e-6)
record('distance_hand_example', 'PASS' if hand_ok else 'FAIL', f"d_cm={res['d_cm'].round(3).tolist()}, nn={res['nn_index'].tolist()}, orient={res['orient_deg'].round(3).tolist()}")

# --- brute force cross-check on real poses (all scenes, variants, 500 random test queries each) -----------------------
worst = 0.0
for (s, v), sp in SPLITS.items():
    q = sp['test']; sel = np.random.default_rng(3).choice(len(q), 500, replace=False)
    r = nearest_view_distance(sp['train'], q.subset(np.isin(np.arange(len(q)), sel)))
    brute = np.linalg.norm(q.centres[np.sort(sel)][:, None, :] - sp['train'].centres[None], axis=2).min(1) * 100
    worst = max(worst, np.abs(r['d_cm'] - brute).max())
record('distance_bruteforce_real', 'PASS' if worst < 1e-9 else 'FAIL', f'KD-tree vs brute force on real poses: max abs diff {worst:.1e} cm')

## 10. Negative controls

*What and why.* A pipeline can produce plausible numbers from wrong inputs. The controls below are built **before** any method is trusted.

* **Control 3 (synthetic pose).** A pose displaced by exactly 3 cm and rotated by exactly 2° must score 3 cm and 2°.
* **Control 5 (dataset integrity).** Train/test disjointness, no missing or duplicated poses, valid rotations, identical frame sets in both ground-truth variants, contiguous frame indices. The loader (restructuring, image/pose/calibration parity, corruption and leakage detection) is unit-tested on a **synthetic fixture** that mimics the Microsoft folder layout. The fixture images are random noise and the fixture exercises code paths only; it says nothing about the real data.
* Controls 1 (mean-pose floor), 2 (shuffled pose) and 4 (oracle) need the baselines and are executed in Section 11.

*Expected output:* all controls pass; the fixture's four negative tests each raise the intended error.

In [ ]:
# ---------------- Control 3: synthetic pose with exactly 3 cm and 2 degrees -----------------------------------------
worst3 = 0.0
for seed in range(200):
    r = np.random.default_rng(seed)
    gt = np.eye(4); gt[:3, :3] = Rotation.random(random_state=seed).as_matrix(); gt[:3, 3] = r.normal(size=3)
    u = r.normal(size=3); u /= np.linalg.norm(u)
    ax = r.normal(size=3); ax /= np.linalg.norm(ax)
    est = gt.copy(); est[:3, 3] = gt[:3, 3] + 0.03 * u                                   # exactly 3 cm
    est[:3, :3] = Rotation.from_rotvec(np.radians(2.0) * ax).as_matrix() @ gt[:3, :3]     # exactly 2 degrees
    worst3 = max(worst3, abs(translation_error_cm(est[:3, 3], gt[:3, 3]) - 3.0).max(), abs(rotation_error_deg(est[:3, :3], gt[:3, :3])[0] - 2.0))
record('control3_synthetic_pose', 'PASS' if worst3 < 1e-6 else 'FAIL', f'200 random poses: max deviation from (3 cm, 2 deg) = {worst3:.1e}')

# ---------------- Control 5 (pose level, real files) -----------------------------------------------------------------
issues = []
for s in CFG['scenes']:
    for sp in ['train', 'test']:
        a, b = POSES[(s, 'dslam', sp)], POSES[(s, 'sfm', sp)]
        if set(a.keys) != set(b.keys): issues.append(f'{s}/{sp}: variants cover different frames')
    for v in CFG['gt_variants']:
        for sp in ['train', 'test']:
            ps = POSES[(s, v, sp)]
            for q in sorted(set(ps.seq)):                        # frames must be contiguous 0..n-1 (nothing silently missing)
                fr = np.sort(ps.frame[ps.seq == q])
                if not np.array_equal(fr, np.arange(len(fr))): issues.append(f'{s}/{v}/{sp}/{q}: {fr.max()+1-len(fr)} frame indices missing')
record('variant_keys_identical', 'PASS' if not [i for i in issues if 'variants' in i] else 'FAIL', 'both ground truths cover exactly the same frames' if not issues else '; '.join(issues[:3]))
record('control5_pose_level', 'PASS' if not issues else 'FAIL',
       'sequence/frame disjointness, contiguous indices, valid rotations, identical frame sets' if not issues else '; '.join(issues[:4]))

In [ ]:
# ---------------- Control 5 (loader): SYNTHETIC FIXTURE in the Microsoft folder layout ----------------------------------
import tempfile
from PIL import Image

def verify_pose_content(scene_dir, expected_by_split, tol=1e-9):
    """Every pose file must equal the table pose with the same key (detects swapped or shifted files)."""
    for split, ps in expected_by_split.items():
        idx = {k: i for i, k in enumerate(ps.keys)}
        for f in (Path(scene_dir) / split / 'poses').iterdir():
            key = f.name[:-len('.pose.txt')]; key = key[:6] + '/' + key[7:]          # 'seq-01-frame-000000' -> 'seq-01/frame-000000'
            assert key in idx, f'pose file without table entry: {key}'
            assert np.abs(np.loadtxt(f) - ps.T[idx[key]]).max() < tol, f'pose content mismatch for {key}'

def assert_no_split_leakage(scene_dir):
    names = {sp: {p.name for p in (Path(scene_dir) / sp / 'rgb').iterdir()} for sp in ['train', 'test']}
    both = names['train'] & names['test']
    assert not both, f'{len(both)} frames appear in BOTH train and test, e.g. {sorted(both)[:2]}'

def build_fixture(tmp, n_per_seq=4, size_hw=(48, 64)):
    rng = np.random.default_rng(123); scene = 'chess'; src = Path(tmp) / 'src' / scene; expected = {}
    src.mkdir(parents=True)
    for split, fn in [('train', 'TrainSplit.txt'), ('test', 'TestSplit.txt')]:
        ps = POSES[('chess', 'dslam', split)]; seqs = sorted(set(ps.seq))
        (src / fn).write_text('\n'.join('sequence' + str(int(q[4:])) for q in seqs) + '\n')
        keys = []
        for q in seqs:
            sel = sorted((k for k in ps.keys if k.startswith(q + '/')), key=lambda k: int(k.split('frame-')[1]))[:n_per_seq]
            (src / q).mkdir(exist_ok=True)
            for k in sel:
                base = k.split('/')[1]
                Image.fromarray(rng.integers(0, 255, (*size_hw, 3), dtype=np.uint8)).save(src / q / f'{base}.color.png')
                write_pose_txt(src / q / f'{base}.pose.txt', ps.T[list(ps.keys).index(k)] @ D_TO_RGB)   # 'original' depth-sensor pose
            keys += sel
        expected[split] = keys
    return Path(tmp) / 'src', expected

def raises(fn):
    try: fn()
    except AssertionError: return True
    except Exception: return False
    return False

fixture_results = {}
with tempfile.TemporaryDirectory() as tmp:
    src_root, expected_keys = build_fixture(tmp)
    exp_sets = {sp: POSES[('chess', 'dslam', sp)].by_key(expected_keys[sp]) for sp in ['train', 'test']}
    out = Path(tmp) / 'out'
    tgt = restructure_scene('chess', src_root, out, 'dslam_cal', link_images=False)
    rep = verify_image_pose_parity(tgt, expect_hw=(48, 64), expect_keys=expected_keys)
    verify_pose_content(tgt, exp_sets)                      # round trip: original @ inv(D_TO_RGB) == dslam c2w
    assert_no_split_leakage(tgt)
    fixture_results['positive_dslam_cal'] = True
    sfm_sets = {sp: POSES[('chess', 'sfm', sp)] for sp in ['train', 'test']}
    tgt2 = restructure_scene('chess', src_root, out, 'sfm_pgt', pgt_poseset_by_split=sfm_sets, link_images=False)
    verify_image_pose_parity(tgt2, expect_hw=(48, 64), expect_keys=expected_keys)
    verify_pose_content(tgt2, {sp: sfm_sets[sp].by_key(expected_keys[sp]) for sp in ['train', 'test']})
    fixture_results['positive_sfm_pgt'] = True

    def fresh():
        d = Path(tmp) / f'neg{len(list(Path(tmp).glob("neg*")))}'; shutil.copytree(tgt, d); return d
    d = fresh(); next((d / 'train' / 'rgb').iterdir()).unlink()
    fixture_results['neg_missing_image_detected'] = raises(lambda: verify_image_pose_parity(d, expect_hw=(48, 64)))
    d = fresh(); next((d / 'test' / 'rgb').iterdir()).write_bytes(b'not an image')
    fixture_results['neg_corrupt_image_detected'] = raises(lambda: verify_image_pose_parity(d, expect_hw=(48, 64)))
    d = fresh(); pf = sorted((d / 'train' / 'poses').iterdir()); a_, b_ = pf[0].read_text(), pf[5].read_text(); pf[0].write_text(b_); pf[5].write_text(a_)
    fixture_results['neg_swapped_poses_detected'] = raises(lambda: verify_pose_content(d, exp_sets))
    d = fresh(); f0 = sorted((d / 'test' / 'rgb').iterdir())[0]
    for sub, suffix in [('rgb', '.color.png'), ('poses', '.pose.txt'), ('calibration', '.calibration.txt')]:
        stem = f0.name[:-len('.color.png')]; shutil.copy(d / 'test' / sub / (stem + suffix), d / 'train' / sub / (stem + suffix))
    fixture_results['neg_leakage_detected'] = raises(lambda: assert_no_split_leakage(d))
print(json.dumps(fixture_results, indent=1))
record('control5_loader_fixture', 'PASS' if all(fixture_results.values()) and len(fixture_results) == 6 else 'FAIL',
       'SYNTHETIC FIXTURE: restructuring round-trips both variants; missing image, corrupt image, swapped poses and train/test leakage are each detected')

In [ ]:
# Archive-extraction test (synthetic archives): a bad CRC on Thumbs.db must be tolerated, a bad CRC on an image must raise.
def _mk(path, members):
    with zipfile.ZipFile(path, 'w') as z:
        for n, d_ in members: z.writestr(n, d_)
def _corrupt(path, member):
    with zipfile.ZipFile(path) as z: i = z.getinfo(member)
    raw = bytearray(Path(path).read_bytes()); off = i.header_offset + 30 + len(i.filename.encode()) + len(i.extra); raw[off] ^= 0xFF
    Path(path).write_bytes(bytes(raw))
def _extract_case(bad_member):
    global LOCAL_ZIP_DIR
    saved = (LOCAL_ZIP_DIR, dict(EXTRACT_REPORT)); t = Path(tempfile.mkdtemp())
    try:
        inner = t / 'seq-01.zip'
        _mk(inner, [('seq-01/frame-000000.color.png', b'img' * 100), ('seq-01/Thumbs.db', b'x' * 500)]); _corrupt(inner, bad_member)
        _mk(t / 'chess.zip', [('chess/seq-01.zip', inner.read_bytes()), ('chess/TrainSplit.txt', b'sequence1\n')])
        LOCAL_ZIP_DIR = t
        try:
            download_scene('chess', t / 'dst'); return 'ok', sorted(p.name for p in (t / 'dst' / 'chess' / 'seq-01').iterdir())
        except RuntimeError: return 'raised', None
    finally:
        LOCAL_ZIP_DIR = saved[0]; EXTRACT_REPORT.clear(); EXTRACT_REPORT.update(saved[1]); shutil.rmtree(t, ignore_errors=True)
r_junk, r_img = _extract_case('seq-01/Thumbs.db'), _extract_case('seq-01/frame-000000.color.png')
print(r_junk, r_img)
record('archive_extraction_policy', 'PASS' if r_junk == ('ok', ['frame-000000.color.png']) and r_img == ('raised', None) else 'FAIL',
       'SYNTHETIC: bad CRC on Thumbs.db is skipped and logged; bad CRC on an image/pose file aborts extraction')

## 11. Mean-pose and oracle baselines (and Controls 1, 2, 4)

*What.* **Mean-pose floor:** every query receives the mean training camera centre and the chordal-mean training rotation. **Oracle:** every query receives the pose of its ground-truth nearest training view. **Shuffled pose:** perfect predictions re-associated with random *other* test frames. *Why.* The floor is the weakest meaningful reference; the oracle is the best any top-1 retrieval can do in translation; the shuffled control proves that mis-ordered pose association produces large errors. *Expected output (stated before running):* floor error ≫ oracle error in translation; shuffled error ≥ floor error; perfect predictions score zero; no other training pose beats the oracle in translation. These are **pose-only measurements** (no image, no localisation method).

*Note on the oracle:* it is selected by translation, so it bounds translation error only; its rotation error is reported but is not a bound.

In [ ]:
def mean_pose_baseline(train, query):
    T = np.tile(np.eye(4), (len(query), 1, 1))
    T[:, :3, :3] = Rotation.from_matrix(train.R).mean().as_matrix()      # chordal L2 mean rotation
    T[:, :3, 3] = train.centres.mean(0)
    return PoseSet(query.keys, T, name='mean_pose')

def oracle_nearest_view(train, query):
    idx = cKDTree(train.centres).query(query.centres)[1]
    return PoseSet(query.keys, train.T[idx], name='oracle'), idx

def derangement(n, seed):
    """Random permutation with no fixed point (a frame is never paired with its own pose)."""
    r = np.random.default_rng(seed); p = r.permutation(n)
    for i in np.where(p == np.arange(n))[0]: j = (i + 1) % n; p[i], p[j] = p[j], p[i]
    assert (p != np.arange(n)).all(); return p

POSE_REF_ROWS, DIST_ROWS = [], []
c1_ok, c2_ok, c4_ok, details = True, True, True, []
for (s, v), sp in SPLITS.items():
    train, test = sp['train'], sp['test']
    floor = score_predictions(mean_pose_baseline(train, test), test)
    orc_ps, nn = oracle_nearest_view(train, test); orc = score_predictions(orc_ps, test)
    perfect = score_predictions(test, test)
    p = derangement(len(test), seed=0)
    shuffled_ps = PoseSet(test.keys, test.T[p], name='shuffled'); shuf = score_predictions(shuffled_ps, test)
    order = np.random.default_rng(5).permutation(len(test))
    reordered = score_predictions(PoseSet(test.keys[order], test.T[order]), test)      # same predictions, rows in different order
    dd = nearest_view_distance(train, test)
    # ---- Control 1: floor is finite and much worse than the oracle in translation
    c1 = np.isfinite(floor.t_err_cm).all() and floor.t_err_cm.median() > 2 * orc.t_err_cm.median()
    # ---- Control 2: perfect -> 0; scoring is order-invariant; shuffled translation error >= floor
    c2 = perfect.t_err_cm.max() < 1e-6 and perfect.r_err_deg.max() < 1e-3 and np.allclose(reordered.t_err_cm.values, perfect.t_err_cm.values) \
         and shuf.t_err_cm.median() >= floor.t_err_cm.median() and shuf.r_err_deg.median() > perfect.r_err_deg.median()
    # ---- Control 4: oracle translation error equals d(q); no training pose is closer (brute force on 300 queries); random picks never beat it
    brute = np.linalg.norm(test.centres[:300, None, :] - train.centres[None], axis=2).min(1) * 100
    rand_idx = np.random.default_rng(9).integers(0, len(train), len(test))
    rand_t = translation_error_cm(train.centres[rand_idx], test.centres)
    c4 = np.abs(orc.t_err_cm.values - dd['d_cm']).max() < 1e-9 and np.abs(brute - orc.t_err_cm.values[:300]).max() < 1e-9 and (rand_t >= orc.t_err_cm.values - 1e-9).all()
    c1_ok &= bool(c1); c2_ok &= bool(c2); c4_ok &= bool(c4)
    details.append(f'{s}/{v}: floor {floor.t_err_cm.median():.1f} cm vs oracle {orc.t_err_cm.median():.1f} cm vs shuffled {shuf.t_err_cm.median():.1f} cm')
    for name, df in [('mean-pose floor', floor), ('shuffled-pose control', shuf), ('oracle nearest view', orc)]:
        POSE_REF_ROWS.append({'scene': s, 'gt': v, 'kind': 'pose-only control (no localisation method)', 'reference': name, **summarise_errors(df)})
    # distance distribution of test queries to the full training map, and of dev queries to the dev map
    dv = nearest_view_distance(sp['dev_map'], sp['dev'])
    for tag, d_ in [('test -> train', dd), ('dev -> dev_map', dv)]:
        DIST_ROWS.append({'scene': s, 'gt': v, 'queries -> map': tag, 'n': len(d_['d_cm']), 'median_d_cm': np.median(d_['d_cm']),
                          'p90_d_cm': np.percentile(d_['d_cm'], 90), 'max_d_cm': d_['d_cm'].max(),
                          'median_orient_deg': np.median(d_['orient_deg']), 'share_orient_gt30_pct': 100 * np.mean(d_['orient_deg'] > CFG['orientation_split_deg'])})
for line_ in details: print(line_)
record('control1_mean_pose_floor', 'PASS' if c1_ok else 'FAIL', 'floor median translation error > 2x oracle in every scene/variant (pose-only)')
record('control2_shuffled_pose', 'PASS' if c2_ok else 'FAIL', 'perfect=0, order-invariant scoring, shuffled translation error >= floor, shuffled rotation error > 0 (pose-only)')
record('control4_oracle_bound', 'PASS' if c4_ok else 'FAIL', 'oracle translation error == d(q); no training pose closer; random picks never beat it (pose-only)')

## 12. NetVLAD feature extraction

*What:* global descriptors from the **official hloc implementation** of NetVLAD (`hloc.extractors.netvlad`): VGG16-NetVLAD-Pitts30K weights, the default whitening layer, 4096-dimensional L2-normalised output. Images are fed at their native 640×480 (hloc's `resize_max` of 1024 is a no-op). One descriptor is computed per image of the training **and** test sequences; descriptors do not depend on the ground-truth variant, so they are computed once and cached (resume-safe, chunked). *Why:* NetVLAD is used exactly as published; nothing is trained or fine-tuned on 7-Scenes. *Expected output:* a cache of 4096-D descriptors per scene and split, plus descriptor sanity checks (dimension, norm, determinism, batch-vs-single agreement).

In [ ]:
NV_CFG = CFG['netvlad']
NV_CACHE = DIRS['cache'] / 'netvlad'; NV_CACHE.mkdir(parents=True, exist_ok=True)

def _nv_paths(scene, split):
    return NV_CACHE / f'{scene}_{split}_keys.json', NV_CACHE / f'{scene}_{split}_desc.npy'

def nv_cache_complete(scene, split):
    kp, dp = _nv_paths(scene, split)
    if not (kp.exists() and dp.exists()): return False
    return json.loads(kp.read_text()) == list(POSES[(scene, 'sfm', split)].keys)

def image_path(scene, key, split):
    """Image of a frame in the restructured scene folder (same images for both ground-truth variants)."""
    return DIRS['data'] / f'{scene}_sfm_pgt' / split / 'rgb' / (key.replace('/', '-') + '.color.png')

NV_MODEL = None
def load_netvlad():
    """Official hloc NetVLAD (VGG16-NetVLAD-Pitts30K, whitening). Loaded lazily, only if some descriptors are missing."""
    global NV_MODEL
    if NV_MODEL is not None: return NV_MODEL
    import torch
    HLOC_DIR = DIRS['ext'] / 'hloc'
    CFG['provenance']['hloc_commit'] = clone_pinned(CFG['urls'].get('hloc_repo', 'https://github.com/cvg/Hierarchical-Localization.git'), HLOC_DIR)
    if str(HLOC_DIR) not in sys.path: sys.path.insert(0, str(HLOC_DIR))
    if LOCAL_ZIP_DIR is not None and (LOCAL_ZIP_DIR / 'Pitts30K_struct.mat').exists():            # optional manual copy of the checkpoint
        dst = Path(torch.hub.get_dir()) / 'netvlad' / 'VGG16-NetVLAD-Pitts30K.mat'; dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists(): shutil.copy(LOCAL_ZIP_DIR / 'Pitts30K_struct.mat', dst)
    from hloc.extractors.netvlad import NetVLAD
    m = NetVLAD({'model_name': NV_CFG['model'], 'whiten': NV_CFG['whiten']}).eval().to(DEVICE)
    ck = Path(torch.hub.get_dir()) / 'netvlad' / (NV_CFG['model'] + '.mat')
    CFG['provenance']['netvlad_checkpoint_sha256'] = hashlib.sha256(ck.read_bytes()).hexdigest()
    NV_MODEL = m; return m

def free_gpu():
    """Release the notebook kernel's GPU memory (NetVLAD weights, cached activations) so the ACE subprocess can use the whole GPU.
    ACE's official 8M-feature training buffer needs most of a 15 GB T4; a NetVLAD model left on the GPU holds ~10 GB."""
    global NV_MODEL
    NV_MODEL = None
    import gc; gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); return round(torch.cuda.mem_get_info()[0] / 2**30, 2)
    except Exception: pass
    return None

def _read_rgb01(path):
    from PIL import Image
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.float32) / 255.0

def nv_describe_paths(paths, batch=None):
    import torch
    from concurrent.futures import ThreadPoolExecutor
    m = load_netvlad(); batch = batch or NV_CFG['batch']; out = []
    with ThreadPoolExecutor(4) as ex, torch.no_grad():
        for i in range(0, len(paths), batch):
            ims = list(ex.map(_read_rgb01, paths[i:i + batch]))
            x = torch.from_numpy(np.stack(ims)).permute(0, 3, 1, 2).contiguous().to(DEVICE)
            out.append(m({'image': x})['global_descriptor'].float().cpu().numpy())
    return np.concatenate(out)

def extract_split(scene, split, chunk=500):
    kp, dp = _nv_paths(scene, split); keys = list(POSES[(scene, 'sfm', split)].keys)
    if nv_cache_complete(scene, split): return
    parts = []
    for ci, i in enumerate(range(0, len(keys), chunk)):
        part = NV_CACHE / f'{scene}_{split}_part{ci:03d}.npy'
        if not part.exists():
            d = nv_describe_paths([image_path(scene, k, split) for k in keys[i:i + chunk]])
            np.save(part, d)
        parts.append(part)
    np.save(dp, np.concatenate([np.load(p) for p in parts])); kp.write_text(json.dumps(keys))
    for p in parts: p.unlink()

NVD = {}     # scene -> {'keys': array, 'desc': (N, 4096) float32, 'index': {key: row}}
def load_descriptors(scene):
    keys, descs = [], []
    for split in ['train', 'test']:
        kp, dp = _nv_paths(scene, split)
        keys += json.loads(kp.read_text()); descs.append(np.load(dp))
    keys = np.array(keys); NVD[scene] = {'keys': keys, 'desc': np.concatenate(descs).astype(np.float32), 'index': {k: i for i, k in enumerate(keys)}}
    return NVD[scene]

def desc_of(scene, keys):
    d = NVD[scene]; return d['desc'][[d['index'][k] for k in keys]]

NV_READY = False
if REAL_IMAGES_AVAILABLE and DEVICE == 'cuda':
    t0 = time.time()
    try:
        for s in CFG['scenes']:
            for sp in ['train', 'test']:
                extract_split(s, sp)
            load_descriptors(s)
        NV_READY = True
        print(f'descriptors ready in {time.time() - t0:.0f} s; provenance:', {k: v for k, v in CFG['provenance'].items() if 'netvlad' in k or 'hloc' in k})
    except Exception as e:
        NV_READY = False
        record('netvlad_extraction', 'FAIL', f'{type(e).__name__}: {str(e)[:300]}')
elif all(nv_cache_complete(s, sp) for s in CFG['scenes'] for sp in ['train', 'test']):
    for s in CFG['scenes']: load_descriptors(s)
    NV_READY = True; print('descriptors loaded from cache (no GPU needed)')
else:
    record('netvlad_extraction', 'NOT_RUN', 'needs the real images and a GPU runtime (or a complete descriptor cache)')

In [ ]:
def nv_sanity():
    d = NVD[CFG['scenes'][0]]; D = d['desc']
    ok_shape = D.shape[1] == NV_CFG['dim'] and np.isfinite(D).all()
    norms = np.linalg.norm(D, axis=1); ok_norm = np.abs(norms - 1).max() < 1e-3
    msg = [f'dim={D.shape[1]}, max |norm-1|={np.abs(norms - 1).max():.1e}']
    ok_det = True
    if NV_MODEL is not None or (REAL_IMAGES_AVAILABLE and DEVICE == 'cuda'):
        sc = CFG['scenes'][0]; ks = list(POSES[(sc, 'sfm', 'train')].keys[::997][:8]); ps = [image_path(sc, k, 'train') for k in ks]
        a = nv_describe_paths(ps, batch=8); b = np.concatenate([nv_describe_paths([p], batch=1) for p in ps]); c = nv_describe_paths(ps, batch=8)
        gap_batch = float(np.abs(a - b).max()); gap_rep = float(np.abs(a - c).max()); gap_cache = float(np.abs(a - desc_of(sc, ks)).max())
        ok_det = gap_batch < 1e-3 and gap_rep < 1e-5 and gap_cache < 1e-3
        msg.append(f'batch-vs-single {gap_batch:.1e}, repeat {gap_rep:.1e}, cache-vs-fresh {gap_cache:.1e}')
    return bool(ok_shape and ok_norm and ok_det), '; '.join(msg)

if NV_READY:
    ok, msg = nv_sanity(); record('netvlad_descriptor_sanity', 'PASS' if ok else 'FAIL', msg)
    record('netvlad_extraction', 'PASS', f'{sum(len(NVD[s]["keys"]) for s in CFG["scenes"])} descriptors, official hloc VGG16-NetVLAD-Pitts30K + whitening')
else:
    record('netvlad_descriptor_sanity', 'NOT_RUN', 'no descriptors')
print('GPU memory free after releasing NetVLAD (GB):', free_gpu())

## 13. NetVLAD top-1 retrieval

*What:* cosine similarity (descriptors are L2-normalised) between each query and every map image; the query receives the pose of the most similar map image. *Why:* this is the retrieval baseline. *Mandatory checks, executed on the real descriptors:* (a) **self-retrieval**: a training image queried against the training map must return itself with pose error ≈ 0; (b) **oracle bound**: no top-1 result can have a smaller translation error than the distance to the nearest training view, `d(q)`, because the returned pose *is* a training pose. *Expected output:* test-sequence predictions and per-frame errors for both ground-truth variants (saved to `results/predictions/`), and the DenseVLAD context table. DenseVLAD is a **different** descriptor; its published medians are a plausibility range for retrieval-type methods, not a NetVLAD target.

In [ ]:
def pose_to_row(T):
    q = Rotation.from_matrix(T[:3, :3]).as_quat()            # x y z w
    return [*T[:3, 3], *q]

def save_predictions(df, name):
    p = DIRS['results/predictions'] / f'{name}.csv'; df.to_csv(p, index=False); return p

def load_pred_poseset(df):
    T = np.tile(np.eye(4), (len(df), 1, 1))
    T[:, :3, :3] = Rotation.from_quat(df[['qx', 'qy', 'qz', 'qw']].values).as_matrix(); T[:, :3, 3] = df[['tx', 'ty', 'tz']].values
    return PoseSet(df['key'].values, T)

def similarity(scene, query_keys, map_keys):
    return desc_of(scene, query_keys) @ desc_of(scene, map_keys).T            # cosine similarity (unit vectors)

def netvlad_top1(scene, map_ps, query_ps):
    S = similarity(scene, query_ps.keys, map_ps.keys)
    nn = S.argmax(1)
    return PoseSet(query_ps.keys, map_ps.T[nn], name='netvlad_top1'), nn, S[np.arange(len(nn)), nn]

def frame_table(pred, gt, extra=None):
    df = score_predictions(pred, gt); p = pred.by_key(gt.keys)
    for j, c in enumerate(['tx', 'ty', 'tz', 'qx', 'qy', 'qz', 'qw']): df[c] = np.array([pose_to_row(T) for T in p.T])[:, j]
    if extra:
        for k, v in extra.items(): df[k] = v
    return df

T1 = {}          # (scene, variant) -> frame table on the TEST sequences
if NV_READY:
    # ---- (a) self-retrieval on the training map (real descriptors, real poses)
    worst_pose, self_hit, minsim = 0.0, 1.0, 1.0
    for s in CFG['scenes']:
        for v in CFG['gt_variants']:
            tr = SPLITS[(s, v)]['train']; pred, nn, sim = netvlad_top1(s, tr, tr)
            hit = np.mean(nn == np.arange(len(tr))); self_hit = min(self_hit, hit); minsim = min(minsim, float(sim.min()))
            e = score_predictions(pred, tr); worst_pose = max(worst_pose, float(e.t_err_cm.max()))
    record('netvlad_self_retrieval', 'PASS' if self_hit > 0.995 and minsim > 1 - 1e-4 and worst_pose < 5.0 else 'FAIL',
           f'self top-1 hit rate (worst scene/variant) {100 * self_hit:.2f} %, min self-similarity {minsim:.6f}, worst self-retrieval translation error {worst_pose:.2f} cm '
           '(any non-self hit must be an exact duplicate image, hence a near-identical pose)')
    # ---- test sequences, both ground truths, + (b) oracle bound
    bound_ok, rows = True, []
    for s in CFG['scenes']:
        for v in CFG['gt_variants']:
            sp = SPLITS[(s, v)]; pred, nn, sim = netvlad_top1(s, sp['train'], sp['test'])
            dd = nearest_view_distance(sp['train'], sp['test'])
            df = frame_table(pred, sp['test'], {'retrieved_key': sp['train'].keys[nn], 'similarity': sim, 'd_nn_cm': dd['d_cm'], 'orient_nn_deg': dd['orient_deg']})
            bound_ok &= bool((df.t_err_cm.values >= dd['d_cm'] - 1e-6).all())
            T1[(s, v)] = df; save_predictions(df, f'netvlad_top1_{s}_{v}')
            rows.append({'scene': s, 'gt': v, 'method': 'NetVLAD top-1', **summarise_errors(df)})
    record('netvlad_oracle_bound', 'PASS' if bound_ok else 'FAIL', 'no top-1 translation error is below the nearest-view distance d(q), for every query, scene and variant')
    NV_TOP1_DF = pd.DataFrame(rows); NV_TOP1_DF.to_csv(DIRS['results/tables'] / 'netvlad_top1_test_summary.csv', index=False)
    print(NV_TOP1_DF.round(2).to_string(index=False))
    # ---- context: DenseVLAD published medians (different descriptor; plausibility range only)
    dv = PUBLISHED_REFERENCE['densevlad_top1_median']; f = CFG['netvlad_sanity_factor']; ctx, ok_ctx = [], True
    for s in CFG['scenes']:
        mt = float(np.mean([summarise_errors(T1[(s, v)])['median_t_cm'] for v in CFG['gt_variants']]))
        lo, hi = dv[s][0] / f, dv[s][0] * f; ok_ctx &= lo <= mt <= hi
        ctx.append({'scene': s, 'NetVLAD top-1 median t (cm, measured, mean of 2 GTs)': round(mt, 1), 'DenseVLAD top-1 median t (cm, PUBLISHED, other descriptor)': dv[s][0],
                    f'plausibility range (x/÷{f:g})': f'{lo:.1f}-{hi:.1f}', 'inside range': lo <= mt <= hi})
    CTX_DF = pd.DataFrame(ctx); print(CTX_DF.to_string(index=False)); CTX_DF.to_csv(DIRS['results/tables'] / 'netvlad_vs_densevlad_context.csv', index=False)
    record('netvlad_sanity_range', 'PASS' if ok_ctx else 'FAIL', 'measured NetVLAD top-1 median translation error lies within a factor ' + f'{f:g} of the published DenseVLAD median (context only; different descriptor)')
else:
    for n in ['netvlad_self_retrieval', 'netvlad_oracle_bound', 'netvlad_sanity_range']: record(n, 'NOT_RUN', 'no descriptors')

## 14. NetVLAD top-k retrieval (constrained descriptor-weight interpolation)

*What:* following the interpolation idea of Sattler et al. (CVPR 2019), the query descriptor `q` is approximated by the `k` most similar map descriptors `d_i` with weights that solve

`min_w ‖q − Σ w_i d_i‖²  subject to  Σ w_i = 1`.

The Lagrangian gives the closed form `w = G⁻¹b + μ·G⁻¹1` with `G = D Dᵀ`, `b = D q`, `μ = (1 − 1ᵀG⁻¹b)/(1ᵀG⁻¹1)`. A tiny ridge term `ε·tr(G)/k·I` (ε = 1e-6) is added to `G` because near-duplicate neighbouring video frames make `G` almost singular; this is a numerical safeguard and a documented deviation, not a tuned parameter. The weights may be negative (only the sum is constrained, as specified).

**Translation:** `c = Σ w_i c_i` over the signed weights (camera centres, metres).
**Rotation:** weighted quaternion average (Markley et al., 2007): the principal eigenvector of `Σ w_i q_i q_iᵀ`. This is invariant to the `q ↔ −q` sign ambiguity and is a valid mean only for **non-negative** weights, so rotations use `w⁺ = max(w, 0)/Σ max(w, 0)` (falling back to the top-1 rotation if all weights are non-positive). Using the same weights for translation and rotation would be incorrect for negative weights; the choice is documented here and counted in the output (share of queries with at least one negative weight).

**k is tuned only on the development sequence** (queries) against the development map (training minus that sequence), over the fixed grid `k ∈ {2, 3, 5, 8, 12, 20}`, by median translation error; the chosen k is then applied once to the test sequences. *Checks:* `k = 1` equals top-1; weights sum to 1; the solver recovers known weights; it agrees with a general-purpose constrained optimiser; the quaternion mean reproduces identical rotations and the geodesic midpoint of two rotations and is sign-invariant.

In [ ]:
def constrained_ls_weights(q, D, ridge=None):
    """min_w ||q - D^T w||^2 s.t. sum(w)=1  (closed form, tiny ridge for conditioning). q: (d,), D: (k, d)."""
    ridge = NV_CFG['ridge'] if ridge is None else ridge
    q = np.asarray(q, np.float64); D = np.asarray(D, np.float64); k = len(D)
    G = D @ D.T; G = G + ridge * np.trace(G) / k * np.eye(k); b = D @ q; one = np.ones(k)
    Gi1 = np.linalg.solve(G, one); Gib = np.linalg.solve(G, b)
    mu = (1.0 - Gib.sum()) / Gi1.sum()
    return Gib + mu * Gi1

def weighted_quat_mean(quats_xyzw, w):
    """Markley et al. weighted average: principal eigenvector of sum_i w_i q_i q_i^T (w >= 0)."""
    Q = np.asarray(quats_xyzw, np.float64); w = np.asarray(w, np.float64)
    M = (Q * w[:, None]).T @ Q
    vals, vecs = np.linalg.eigh(M); q = vecs[:, -1]
    ref = Q[np.argmax(w)]
    return q if q @ ref >= 0 else -q

def interpolate_pose(Tk, w):
    """Tk: (k,4,4) camera-to-world poses of the retrieved images, w: signed weights summing to 1."""
    T = np.eye(4); T[:3, 3] = (w[:, None] * Tk[:, :3, 3]).sum(0)
    wp = np.clip(w, 0, None)
    if wp.sum() <= 1e-12: wp = np.zeros_like(w); wp[np.argmax(w)] = 1.0
    wp = wp / wp.sum()
    T[:3, :3] = Rotation.from_quat(weighted_quat_mean(Rotation.from_matrix(Tk[:, :3, :3]).as_quat(), wp)).as_matrix()
    return T

def netvlad_topk(scene, map_ps, query_ps, k):
    Dm = desc_of(scene, map_ps.keys); Dq = desc_of(scene, query_ps.keys)
    S = Dq @ Dm.T
    idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    idx = np.take_along_axis(idx, np.argsort(-np.take_along_axis(S, idx, 1), axis=1), 1)      # sorted by similarity
    T = np.empty((len(query_ps), 4, 4)); W = np.empty((len(query_ps), k))
    for i in range(len(query_ps)):
        W[i] = constrained_ls_weights(Dq[i], Dm[idx[i]]) if k > 1 else np.ones(1)
        T[i] = interpolate_pose(map_ps.T[idx[i]], W[i])
    return PoseSet(query_ps.keys, T, name=f'netvlad_top{k}'), {'idx': idx, 'W': W}

# ------------------------- unit tests of the interpolation machinery (synthetic data) ---------------------------
rng = np.random.default_rng(7)
Dsyn = rng.normal(size=(6, 64)); Dsyn /= np.linalg.norm(Dsyn, axis=1, keepdims=True)
w_true = np.array([0.5, 0.3, 0.4, -0.1, -0.2, 0.1]); w_true /= w_true.sum()
q_syn = w_true @ Dsyn                                                        # query that IS an affine combination
w_est = constrained_ls_weights(q_syn, Dsyn, ridge=0.0)
t_rec = np.abs(w_est - w_true).max()
q_off = rng.normal(size=64); w_off = constrained_ls_weights(q_off, Dsyn, ridge=0.0)
from scipy.optimize import minimize
res = minimize(lambda w: np.sum((q_off - w @ Dsyn) ** 2), np.ones(6) / 6, constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}], method='SLSQP', options={'ftol': 1e-14, 'maxiter': 500})
t_opt = np.abs(w_off - res.x).max()
t_sum = abs(w_off.sum() - 1)
Ra = Rotation.from_euler('xyz', [10, 20, 30], degrees=True); Rb = Rotation.from_euler('xyz', [40, -10, 5], degrees=True)
qa, qb = Ra.as_quat(), Rb.as_quat()
q_same = weighted_quat_mean(np.stack([qa, -qa, qa]), np.array([0.2, 0.5, 0.3]))                  # identical rotation, mixed signs
t_same = np.degrees((Rotation.from_quat(q_same).inv() * Ra).magnitude())
q_mid = weighted_quat_mean(np.stack([qa, qb]), np.array([0.5, 0.5]))
from scipy.spatial.transform import Slerp
t_mid = np.degrees((Rotation.from_quat(q_mid).inv() * Slerp([0, 1], Rotation.concatenate([Ra, Rb]))([0.5])[0]).magnitude())
q_flip = weighted_quat_mean(np.stack([qa, -qb]), np.array([0.5, 0.5]))
t_flip = np.degrees((Rotation.from_quat(q_flip).inv() * Rotation.from_quat(q_mid)).magnitude())
# k = 1 pose equals the retrieved pose; interpolation with weights (1,0,...) equals that pose
Tk = np.stack([np.eye(4)] * 3); Tk[:, :3, :3] = Rotation.concatenate([Ra, Rb, Ra]).as_matrix(); Tk[:, :3, 3] = [[0, 0, 0], [1, 0, 0], [0, 2, 0]]
Tint = interpolate_pose(Tk, np.array([1.0, 0.0, 0.0])); t_k1 = max(np.abs(Tint[:3, 3] - Tk[0, :3, 3]).max(), np.abs(Tint[:3, :3] - Tk[0, :3, :3]).max())
Tneg = interpolate_pose(Tk, np.array([1.5, -0.5, 0.0]))                                           # negative weight: translation extrapolates, rotation stays valid
ok_neg = np.allclose(Tneg[:3, 3], [-0.5, 0, 0]) and abs(np.linalg.det(Tneg[:3, :3]) - 1) < 1e-9
ok = t_rec < 1e-8 and t_opt < 1e-5 and t_sum < 1e-12 and t_same < 1e-6 and t_mid < 1e-6 and t_flip < 1e-6 and t_k1 < 1e-9 and ok_neg
record('netvlad_topk_unit_tests', 'PASS' if ok else 'FAIL',
       f'weight recovery {t_rec:.1e}; vs SLSQP {t_opt:.1e}; sum-1 {t_sum:.1e}; quat mean identical {t_same:.1e} deg, midpoint vs slerp {t_mid:.1e} deg, sign flip {t_flip:.1e} deg; k=1 {t_k1:.1e}; negative-weight pose valid={ok_neg}')

In [ ]:
TOPK, K_STAR, DEV_TABLE = {}, {}, []
if NV_READY:
    # ---- consistency: k = 1 reproduces top-1 on the real test descriptors
    s0, v0 = CFG['scenes'][0], CFG['gt_variants'][0]; sp0 = SPLITS[(s0, v0)]
    p1, _, _ = netvlad_top1(s0, sp0['train'], sp0['test']); pk1, _ = netvlad_topk(s0, sp0['train'], sp0['test'], 1)
    record('netvlad_topk_k1_equals_top1', 'PASS' if np.abs(p1.T - pk1.T).max() < 1e-9 else 'FAIL', 'k = 1 interpolation reproduces top-1 retrieval on real descriptors')
    # ---- tune k on the DEVELOPMENT sequence only
    def _tune(s, v):
        sp = SPLITS[(s, v)]; rows = []
        for k in NV_CFG['k_grid']:
            pred, info = netvlad_topk(s, sp['dev_map'], sp['dev'], k); e = score_predictions(pred, sp['dev'])
            rows.append({'scene': s, 'gt': v, 'k': k, 'dev_median_t_cm': float(e.t_err_cm.median()), 'dev_median_r_deg': float(e.r_err_deg.median()),
                         'dev_acc_5cm5deg_pct': summarise_errors(e)['acc_5cm_5deg_pct'], 'share_queries_with_negative_weight_pct': float(100 * (info['W'] < 0).any(1).mean())})
        return rows
    for s in CFG['scenes']:
        for v in CFG['gt_variants']:
            rows = cached_json(DIRS['cache'] / f'netvlad_dev_tuning_{s}_{v}.json', lambda s=s, v=v: _tune(s, v))
            dfk = pd.DataFrame(rows); DEV_TABLE.append(dfk)
            best = dfk.sort_values(['dev_median_t_cm', 'k']).iloc[0]; K_STAR[(s, v)] = int(best.k)
    DEV_DF = pd.concat(DEV_TABLE); DEV_DF.to_csv(DIRS['results/tables'] / 'netvlad_topk_dev_tuning.csv', index=False)
    print('development-sequence tuning (test sequences untouched):'); print(DEV_DF.round(2).to_string(index=False)); print('chosen k:', K_STAR)
    if any(k == max(NV_CFG['k_grid']) for k in K_STAR.values()): print('NOTE: a chosen k is at the upper edge of the pre-registered grid; the grid is not extended after seeing results (it is reported as a limitation).')
    log_event('netvlad_k_star', K_STAR={f'{a}/{b}': c for (a, b), c in K_STAR.items()})
    # ---- apply the chosen k once to the test sequences
    rows = []
    for s in CFG['scenes']:
        for v in CFG['gt_variants']:
            sp = SPLITS[(s, v)]; k = K_STAR[(s, v)]; pred, info = netvlad_topk(s, sp['train'], sp['test'], k)
            dd = nearest_view_distance(sp['train'], sp['test'])
            df = frame_table(pred, sp['test'], {'k': k, 'min_weight': info['W'].min(1), 'd_nn_cm': dd['d_cm'], 'orient_nn_deg': dd['orient_deg']})
            TOPK[(s, v)] = df; save_predictions(df, f'netvlad_topk_{s}_{v}')
            rows.append({'scene': s, 'gt': v, 'method': f'NetVLAD top-k (k={k})', **summarise_errors(df), 'share_negative_weight_pct': float(100 * (info['W'] < 0).any(1).mean())})
    NV_TOPK_DF = pd.DataFrame(rows); NV_TOPK_DF.to_csv(DIRS['results/tables'] / 'netvlad_topk_test_summary.csv', index=False)
    print(NV_TOPK_DF.round(2).to_string(index=False))
    record('netvlad_topk_dev_only_tuning', 'PASS', f'k chosen on the development sequence only: {dict((f"{a}/{b}", c) for (a, b), c in K_STAR.items())}')
else:
    for n in ['netvlad_topk_k1_equals_top1', 'netvlad_topk_dev_only_tuning']: record(n, 'NOT_RUN', 'no descriptors')

## 15. ACE setup and integration

*What:* the **official Niantic ACE** repository (`nianticlabs/ace`) is cloned at a recorded commit; its OpenCV-dependent C++ solver `dsacstar` (the official RANSAC / PnP pose solver) is compiled; the pretrained encoder shipped in the repository is checked. Three minimal, documented deviations from the official code, none of which changes the method:

1. **Build configuration only.** The official `dsacstar/setup.py` expects a conda environment; `setup_p22.py` builds the *same* sources with the system OpenCV headers.
2. **Seed.** The official trainer hard-codes `base_seed = 2089` and has no seed argument, but the study requires seeds 0, 1 and 2. A one-line patch reads the seed from the environment variable `P22_SEED` (default: the official 2089 when unset).
3. **PyTorch compatibility.** The official trainer reads `optimizer._step_count`, a private attribute that current PyTorch no longer provides (the run crashed with `AttributeError`). The patch counts the optimiser steps itself, which is what that attribute counted (GradScaler may skip a step); learning-rate scheduling is unchanged.

All patches are listed in `ACE_PATCHES` and saved as a diff (`ace_trainer_patches.diff`) next to the logs.

*Checks:* the solver is run on **synthetic scene coordinates** generated from a known pose (it must recover that pose, and report an inlier count); ACE's own dataset reader is run on the restructured scene folders and must return exactly the ground-truth poses of the pose files and the stored focal length. *Expected output:* a compiled solver, a patched trainer, and passing checks. If any step fails, the exact error is recorded; no alternative method is substituted.

In [ ]:
ACE_DIR = DIRS['ext'] / 'ace'
ACE_READY = False
def sh(cmd, cwd=None, env=None, timeout=None, log=None):
    r = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True, timeout=timeout)
    if log: Path(log).write_text((r.stdout or '') + '\n--- stderr ---\n' + (r.stderr or ''))
    return r

ACE_PATCHES = [
    # (1) seed: the official trainer hard-codes 2089 and has no seed argument; the study needs seeds 0, 1, 2 (default stays 2089)
    ('self.base_seed = 2089', "self.base_seed = int(__import__('os').environ.get('P22_SEED', 2089))"),
    # (2) PyTorch compatibility, behaviour unchanged: the official code reads optimizer._step_count, a private attribute that current PyTorch
    #     no longer sets. We count the optimiser steps that actually happen (GradScaler may skip a step) with a thin wrapper, which is
    #     exactly what the old attribute counted.
    ('                                                       cycle_momentum=False)',
     '                                                       cycle_momentum=False)\n'
     '        # P22: count the optimiser steps that actually happen (wrapped AFTER the scheduler has installed its own step tracker)\n'
     '        self._p22_opt_steps = 0\n'
     '        _p22_orig_step = self.optimizer.step\n'
     '        def _p22_counting_step(*a, **k):\n'
     '            self._p22_opt_steps += 1\n'
     '            return _p22_orig_step(*a, **k)\n'
     '        _p22_counting_step._wrapped_by_lr_sched = True\n'
     '        self.optimizer.step = _p22_counting_step'),
    ('old_optimizer_step = self.optimizer._step_count', 'old_optimizer_step = self._p22_opt_steps'),
    ('if old_optimizer_step < self.optimizer._step_count < self.scheduler.total_steps:', 'if old_optimizer_step < self._p22_opt_steps < self.scheduler.total_steps:'),
]

def setup_ace():
    CFG['provenance']['ace_repo_commit'] = clone_pinned(CFG['urls']['ace_repo'], ACE_DIR)
    # --- (1) system dependencies for the C++ solver
    if shutil.which('apt-get') and not Path('/usr/include/opencv4/opencv2').exists():
        sh(['apt-get', 'update', '-q']); r = sh(['apt-get', 'install', '-y', '-q', 'libopencv-dev'])
        assert r.returncode == 0, 'apt-get install libopencv-dev failed: ' + r.stderr[-300:]
    _pip(['scikit-image', 'trimesh', 'pyrender'] + ([] if importlib.util.find_spec('cv2') else ['opencv-python-headless']))
    # --- (2) documented patches to ace_trainer.py (see ACE_PATCHES); the diff is saved next to the logs
    tr = ACE_DIR / 'ace_trainer.py'; src = tr.read_text()
    if 'P22_SEED' not in src:
        orig = src
        for old, new_ in ACE_PATCHES:
            assert old in src, f'patch target not found (ACE version changed?): {old[:60]!r}'
            src = src.replace(old, new_)
        tr.write_text(src)
        import difflib
        (DIRS['results/logs'] / 'ace_trainer_patches.diff').write_text(''.join(difflib.unified_diff(orig.splitlines(1), src.splitlines(1), 'ace_trainer.py (official)', 'ace_trainer.py (P22 patches)')))
    # --- (3) build dsacstar (documented deviation 1)
    dsac = ACE_DIR / 'dsacstar'
    if not list(dsac.glob('dsacstar*.so')):
        (dsac / 'setup_p22.py').write_text("""from setuptools import setup
from torch.utils.cpp_extension import CppExtension, BuildExtension
import os
inc = os.environ.get('P22_OPENCV_INC', '/usr/include/opencv4'); lib = os.environ.get('P22_OPENCV_LIB', '/usr/lib/x86_64-linux-gnu')
setup(name='dsacstar', ext_modules=[CppExtension(name='dsacstar', sources=['dsacstar.cpp', 'thread_rand.cpp'], include_dirs=[inc], library_dirs=[lib],
      libraries=['opencv_core', 'opencv_calib3d'], extra_compile_args=['-fopenmp'])], cmdclass={'build_ext': BuildExtension})
""")
        r = sh([sys.executable, 'setup_p22.py', 'build_ext', '--inplace'], cwd=dsac, log=DIRS['results/logs'] / 'dsacstar_build.log')
        assert list(dsac.glob('dsacstar*.so')), 'dsacstar build failed, see dsacstar_build.log: ' + (r.stderr or '')[-400:]
    # --- (4) pretrained encoder shipped in the official repository
    enc = ACE_DIR / 'ace_encoder_pretrained.pt'
    assert enc.exists(), 'pretrained encoder missing from the ACE repository'
    CFG['provenance']['ace_encoder_sha256'] = hashlib.sha256(enc.read_bytes()).hexdigest()
    return dsac

def ace_env(seed=None):
    env = dict(os.environ); env['PYTHONPATH'] = os.pathsep.join([str(ACE_DIR / 'dsacstar'), str(ACE_DIR), env.get('PYTHONPATH', '')])
    if seed is not None: env['P22_SEED'] = str(seed)
    return env

if REAL_IMAGES_AVAILABLE and DEVICE == 'cuda':
    try:
        setup_ace(); ACE_READY = True
        record('ace_build', 'PASS', f'official ACE at {CFG["provenance"]["ace_repo_commit"][:8]}; dsacstar built; encoder sha256 {CFG["provenance"]["ace_encoder_sha256"][:12]}')
    except Exception as e:
        record('ace_build', 'FAIL', f'{type(e).__name__}: {str(e)[:400]}')
else:
    record('ace_build', 'NOT_RUN', 'needs the real images and a CUDA GPU runtime')

In [ ]:
SOLVER_TEST = r"""
import sys, json, numpy as np, torch
import dsacstar
from scipy.spatial.transform import Rotation as R
H, W, S, f, px, py = 480, 640, 8, 525., 320., 240.
res = []
for seed, noise in [(0, 0.0), (1, 0.0), (2, 0.02)]:
    r = np.random.default_rng(seed); T = np.eye(4); T[:3, :3] = R.random(random_state=seed).as_matrix(); T[:3, 3] = r.normal(size=3)
    ys, xs = np.meshgrid(np.arange(H // S), np.arange(W // S), indexing='ij'); u = xs * S + S / 2; v = ys * S + S / 2
    d = 1.5 + 0.5 * np.sin(xs / 7.) + 0.5 * np.cos(ys / 5.)
    cam = np.stack([(u - px) / f * d, (v - py) / f * d, d, np.ones_like(d)], 0).reshape(4, -1)
    world = (T @ cam)[:3].reshape(3, H // S, W // S) + noise * r.normal(size=(3, H // S, W // S))
    out = torch.zeros(4, 4)
    inl = dsacstar.forward_rgb(torch.tensor(world, dtype=torch.float32).unsqueeze(0), out, 64, 10., f, px, py, 100., 100., S)
    o = out.numpy()
    res.append({'noise': noise, 't_cm': float(np.linalg.norm(o[:3, 3] - T[:3, 3]) * 100), 'r_deg': float(np.degrees(R.from_matrix(o[:3, :3].T @ T[:3, :3]).magnitude())), 'inliers': float(inl)})
print('RESULT_JSON' + json.dumps(res))
"""
DATASET_TEST = r"""
import sys, json, numpy as np
from pathlib import Path
from dataset import CamLocDataset
scene, split = sys.argv[1], sys.argv[2]
ds = CamLocDataset(Path(scene) / split, mode=0, image_height=480)
out = {'n': len(ds), 'items': []}
for i in [0, len(ds) // 2, len(ds) - 1]:
    it = ds[i]; out['items'].append({'i': i, 'file': str(it[-1]), 'pose': np.asarray(it[2]).tolist(), 'K': np.asarray(it[4]).tolist(), 'img_shape': list(np.asarray(it[0]).shape)})
print('RESULT_JSON' + json.dumps(out))
"""
def run_json(code_str, args=(), cwd=None, env=None):
    p = DIRS['cache'] / '_snippet.py'; p.write_text(code_str)
    r = sh([sys.executable, str(p), *map(str, args)], cwd=cwd, env=env, timeout=1800)
    line = [l for l in r.stdout.splitlines() if l.startswith('RESULT_JSON')]
    if not line: raise RuntimeError('snippet failed: ' + (r.stderr or r.stdout)[-600:])
    return json.loads(line[-1][len('RESULT_JSON'):])

ACE_TAGS = {'dslam': 'dslam_cal', 'sfm': 'sfm_pgt'}
def ace_scene_dir(scene, variant, dtag=''):
    """Scene folder in ACE layout. dtag '' = full training density; otherwise a thinned-map folder (Section 20)."""
    return DIRS['data'] / (f'{scene}_{ACE_TAGS[variant]}' + (f'_{dtag}' if dtag else ''))
DENS_MAPS = {}      # (scene, variant, dtag) -> thinned training PoseSet (filled in Section 20)
def map_for(scene, variant, dtag=''): return SPLITS[(scene, variant)]['train'] if not dtag else DENS_MAPS[(scene, variant, dtag)]

if ACE_READY:
    sres = run_json(SOLVER_TEST, env=ace_env())
    print(sres)
    ok_solver = all(r['t_cm'] < 1e-3 and r['r_deg'] < 1e-3 for r in sres if r['noise'] == 0) and all(r['t_cm'] < 2 and r['r_deg'] < 0.5 for r in sres if r['noise'] > 0) and all(r['inliers'] > 100 for r in sres)
    record('ace_solver_synthetic', 'PASS' if ok_solver else 'FAIL', f'official dsacstar RANSAC on synthetic scene coordinates: noise-free error {max(r["t_cm"] for r in sres if r["noise"] == 0):.1e} cm; 2 cm coordinate noise -> {sres[-1]["t_cm"]:.2f} cm / {sres[-1]["r_deg"]:.2f} deg')
    # ACE's own dataset reader on our restructured folders, both variants
    worst_t, worst_f, n_ok = 0.0, 0.0, True
    for s in CFG['scenes']:
        for v in CFG['gt_variants']:
            for sp in ['train', 'test']:
                out = run_json(DATASET_TEST, args=[ace_scene_dir(s, v), sp], cwd=ACE_DIR, env=ace_env())
                gt = POSES[(s, v, sp)]; n_ok &= out['n'] == len(gt)
                for it in out['items']:
                    name = Path(it['file']).name[:-len('.color.png')]; key = name[:6] + '/' + name[7:]
                    Tg = gt.T[list(gt.keys).index(key)]
                    worst_t = max(worst_t, float(np.linalg.norm(np.array(it['pose'])[:3, 3] - Tg[:3, 3]) * 100))
                    worst_f = max(worst_f, abs(it['K'][0][0] - (gt.focal[list(gt.keys).index(key)] if v == 'sfm' else CFG['rgb_focal_default'])))
    record('ace_dataset_parity', 'PASS' if n_ok and worst_t < 0.05 and worst_f < 1e-3 else 'FAIL',
           f'ACE CamLocDataset returns the pose-repository poses (max centre difference {worst_t:.3f} cm) and the stored focal length (max diff {worst_f:.1e}); frame counts match={n_ok}')
else:
    for n in ['ace_solver_synthetic', 'ace_dataset_parity']: record(n, 'NOT_RUN', 'ACE not built in this environment')

## 16. ACE training

*What:* the official `train_ace.py` is run unchanged (default hyper-parameters: 16 epochs, frozen pretrained encoder, scene head only) once per **(scene, ground-truth variant, seed)**; the head weights, the training log and the run configuration are stored under `ace_runs/` (Google Drive on Colab), so a finished run is never repeated after a disconnect. The two ground-truth variants are trained separately on their own pose files (`<scene>_dslam_cal` = original D-SLAM poses calibrated to the RGB sensor, `<scene>_sfm_pgt` = SfM pseudo ground truth); the only difference between them is the pose files. The training buffer keeps the official 8 M features; **only if the GPU runs out of memory** the run is repeated once with a 4 M buffer and the fallback is recorded in `run.json` (a documented deviation that affects only that run). *Expected output:* one `head.pt` per run.

In [ ]:
def run_dir(scene, variant, seed, dtag=''): return DIRS['runs'] / (f'{scene}_{ACE_TAGS[variant]}' + (f'_{dtag}' if dtag else '')) / f'seed{seed}'

def train_ace(scene, variant, seed, dtag=''):
    """Official train_ace.py. Resume-safe: a run with a finished marker is skipped."""
    rd = run_dir(scene, variant, seed, dtag); rd.mkdir(parents=True, exist_ok=True)
    head, marker = rd / 'head.pt', rd / 'train.done'
    if marker.exists() and head.exists(): return json.loads((rd / 'run.json').read_text())
    info = {'scene': scene, 'variant': variant, 'seed': seed, 'density_tag': dtag or 'full', 'scene_dir': str(ace_scene_dir(scene, variant, dtag)), 'ace_commit': CFG['provenance'].get('ace_repo_commit'),
            'buffer': CFG['ace']['buffer_default'], 'deviations': ['seed via P22_SEED (official default 2089)', 'dsacstar built with setup_p22.py (system OpenCV)', 'optimizer step counter patch for current PyTorch (behaviour unchanged)']}
    for attempt, buf in enumerate([CFG['ace']['buffer_default'], CFG['ace']['buffer_fallback']]):
        cmd = [sys.executable, 'train_ace.py', str(ace_scene_dir(scene, variant, dtag)), str(head)] + ([] if attempt == 0 else ['--training_buffer_size', str(buf)])
        info['gpu_free_gb_before_launch'] = free_gpu()
        if info['gpu_free_gb_before_launch'] is not None and info['gpu_free_gb_before_launch'] < 12 and attempt == 0:
            print(f'WARNING: only {info["gpu_free_gb_before_launch"]} GB of GPU memory free; the official 8M buffer may not fit (automatic fallback to 4M on OOM)')
        t0 = time.time(); r = sh(cmd, cwd=ACE_DIR, env=ace_env(seed), log=rd / 'train.log')
        info.update({'buffer': buf, 'train_seconds': round(time.time() - t0, 1), 'returncode': r.returncode, 'cmd': cmd[1:]})
        if r.returncode == 0 and head.exists(): break
        if attempt == 0 and 'out of memory' in ((r.stdout or '') + (r.stderr or '')).lower():
            info['deviations'].append(f'CUDA out of memory with the official 8M buffer -> retried with {CFG["ace"]["buffer_fallback"]}'); continue
        raise RuntimeError(f'train_ace.py failed for {scene}/{variant}/seed{seed} (see {rd / "train.log"}): ' + ((r.stderr or r.stdout)[-500:]))
    (rd / 'run.json').write_text(json.dumps(info, indent=1)); marker.write_text('ok')
    log_event('ace_train_done', **{k: info[k] for k in ['scene', 'variant', 'seed', 'density_tag', 'train_seconds', 'buffer']})
    return info

ACE_PLAN = [(s, v, sd) for (s, v) in [('chess', 'sfm')] + [(s, v) for s in CFG['scenes'] for v in CFG['gt_variants'] if (s, v) != ('chess', 'sfm')] for sd in CFG['ace']['seeds_to_run']]
# the first planned run (chess, SfM, seed 0) doubles as the smoke test for the reproduction gate in Section 17
print(len(ACE_PLAN), 'ACE runs planned (scene, ground truth, seed):', ACE_PLAN[:3], '...')

## 17. ACE inference, scoring and reproduction

*What:* the official `test_ace.py` (official RANSAC solver `dsacstar.forward_rgb`, 64 hypotheses, 10 px inlier threshold, defaults) is run on the official **test sequences** for every trained head. It writes one line per frame: the estimated pose (world→camera quaternion + translation), ACE's own rotation and translation errors, and the **inlier count**. The notebook then
1. converts the estimated poses to camera→world and scores them with the Section-8 metrics against the pose repository's ground truth of the same variant (by key);
2. **parity check:** recomputes the errors ACE logged itself (ACE's rotation error is `cv2.Rodrigues` of `R̂Rᵀ`, ours is the trace formula) and requires agreement;
3. stores per-frame results (`ace_<scene>_<gt>_seed<k>.csv`: errors, inlier count, nearest-view distance `d(q)` and orientation difference) and the per-seed summary;
4. **reproduction check** against the published ACE numbers (kept in `PUBLISHED_REFERENCE`, never mixed with measured values): SfM Chess within 1 percentage point of 100 %, Stairs within 3 points of 81.9 %, on the mean of the three seeds. D-SLAM references are approximate (third-party) and only reported.

A **smoke gate** protects the compute budget: the first run (Chess, SfM, seed 0) is trained and scored alone, and if it reaches less than 90 % accuracy the remaining runs are not started and the investigation list below is printed. *Expected output:* per-seed tables and the reproduction table.

In [ ]:
def test_ace(scene, variant, seed, dtag=''):
    """Official test_ace.py on the test sequences; resume-safe."""
    rd = run_dir(scene, variant, seed, dtag); head = rd / 'head.pt'; sc_dir = ace_scene_dir(scene, variant, dtag)
    plog = rd / f'poses_{sc_dir.name}_seed{seed}.txt'; n_test = len(POSES[(scene, variant, 'test')])
    if plog.exists() and len(plog.read_text().splitlines()) == n_test: return plog
    r = sh([sys.executable, 'test_ace.py', str(sc_dir), str(head), '--session', f'seed{seed}'], cwd=ACE_DIR, env=ace_env(seed), log=rd / 'test.log')
    if r.returncode != 0 or not plog.exists() or len(plog.read_text().splitlines()) != n_test:
        raise RuntimeError(f'test_ace.py failed for {scene}/{variant}/seed{seed} (see {rd / "test.log"}): ' + ((r.stderr or r.stdout)[-500:]))
    lines = (rd / 'test.log').read_text().splitlines(); (rd / 'test.log').write_text('\n'.join(lines[:50] + ['...'] + lines[-50:]))      # keep the log small
    return plog

def parse_ace_poses(path):
    """test_ace.py line: name qw qx qy qz tx ty tz r_err(deg) t_err(m) inliers; the pose is WORLD->CAMERA."""
    rows = [l.split() for l in Path(path).read_text().splitlines() if l.strip()]
    keys, T, ace_r, ace_t, inl = [], [], [], [], []
    for p in rows:
        name = p[0][:-len('.color.png')]; keys.append(name[:6] + '/' + name[7:])
        q = np.array([float(p[2]), float(p[3]), float(p[4]), float(p[1])]); t = np.array([float(p[5]), float(p[6]), float(p[7])])
        Rw2c = Rotation.from_quat(q).as_matrix(); Tc2w = np.eye(4); Tc2w[:3, :3] = Rw2c.T; Tc2w[:3, 3] = -Rw2c.T @ t; T.append(Tc2w)
        ace_r.append(float(p[8])); ace_t.append(float(p[9]) * 100); inl.append(float(p[10]))
    return PoseSet(keys, np.stack(T)), np.array(ace_r), np.array(ace_t), np.array(inl)

def score_ace_run(scene, variant, seed, plog, dtag=''):
    gt = SPLITS[(scene, variant)]['test']; pred, ace_r, ace_t, inl = parse_ace_poses(plog)
    assert set(pred.keys) == set(gt.keys), 'ACE output does not cover exactly the test frames'
    order = {k: i for i, k in enumerate(pred.keys)}; ix = [order[k] for k in gt.keys]
    pred = pred.by_key(gt.keys); ace_r, ace_t, inl = ace_r[ix], ace_t[ix], inl[ix]
    dd = nearest_view_distance(map_for(scene, variant, dtag), gt)
    df = frame_table(pred, gt, {'inliers': inl, 'ace_t_err_cm': ace_t, 'ace_r_err_deg': ace_r, 'd_nn_cm': dd['d_cm'], 'orient_nn_deg': dd['orient_deg']})
    par_t = float(np.abs(df.t_err_cm - df.ace_t_err_cm).max()); par_r = float(np.abs(df.r_err_deg - df.ace_r_err_deg).max())
    return df, par_t, par_r

ACE_RES, ACE_PARITY = {}, {}
ACE_DENS_RES, ACE_DENS_PARITY = {}, {}
def run_and_score(scene, variant, seed, dtag=''):
    """Train + test + score one ACE run. dtag '' is full density (results in ACE_RES); thinned runs go to ACE_DENS_RES (Section 20)."""
    train_ace(scene, variant, seed, dtag); plog = test_ace(scene, variant, seed, dtag)
    df, par_t, par_r = score_ace_run(scene, variant, seed, plog, dtag)
    if dtag:
        ACE_DENS_RES[(scene, variant, dtag, seed)] = df; ACE_DENS_PARITY[(scene, variant, dtag, seed)] = (par_t, par_r)
        save_predictions(df, f'ace_{scene}_{variant}_{dtag}_seed{seed}')
    else:
        ACE_RES[(scene, variant, seed)] = df; ACE_PARITY[(scene, variant, seed)] = (par_t, par_r)
        save_predictions(df, f'ace_{scene}_{variant}_seed{seed}')
    return df

def load_finished_runs():
    for (s, v, sd) in ACE_PLAN:
        plog = run_dir(s, v, sd) / f'poses_{ace_scene_dir(s, v).name}_seed{sd}.txt'
        if (s, v, sd) not in ACE_RES and plog.exists() and len(plog.read_text().splitlines()) == len(POSES[(s, v, 'test')]):
            df, pt, pr = score_ace_run(s, v, sd, plog); ACE_RES[(s, v, sd)] = df; ACE_PARITY[(s, v, sd)] = (pt, pr)

INVESTIGATE = """Suspiciously low ACE accuracy. Investigate before continuing (do not tune on the test set):
 1. ace_dataset_parity and the D-SLAM/SfM pose checks in Sections 7-8 (pose convention, focal length);
 2. ace_runs/<run>/train.log: did the loss decrease, was the 8M buffer reduced, any CUDA warnings?
 3. torch / CUDA versions versus the official environment (PyTorch 2.0, CUDA 11.8) - newer builds change numerics;
 4. test.log and the parity numbers: are ACE's own errors consistent with ours?
 5. seed and GPU nondeterminism (compare seed 1)."""

ACE_BUDGET_MIN = 360          # wall-clock budget for one execution of this cell; every finished run is stored, so a re-run resumes
if ACE_READY:
    load_finished_runs(); t_start = time.time(); gate_ok = True
    first = ACE_PLAN[0]
    if first not in ACE_RES:
        print('smoke run (chess, SfM, seed 0): training + inference ...'); run_and_score(*first)
    acc0 = summarise_errors(ACE_RES[first])['acc_5cm_5deg_pct']; gate = CFG['ace']['smoke_gate_min_acc_pct']
    gate_ok = acc0 >= gate
    record('ace_smoke_gate', 'PASS' if gate_ok else 'FAIL', f'chess/SfM/seed0 accuracy {acc0:.1f} % (gate {gate:g} %)')
    if not gate_ok: print(INVESTIGATE)
    else:
        for run in ACE_PLAN[1:]:
            if run in ACE_RES: continue
            if (time.time() - t_start) / 60 > ACE_BUDGET_MIN:
                print(f'time budget of {ACE_BUDGET_MIN} min used up; re-run this cell to resume ({len(ACE_RES)}/{len(ACE_PLAN)} runs done)'); break
            t0 = time.time(); df = run_and_score(*run)
            print(f'{run}: acc {summarise_errors(df)["acc_5cm_5deg_pct"]:.1f} %  median {df.t_err_cm.median():.2f} cm / {df.r_err_deg.median():.2f} deg  ({(time.time() - t0) / 60:.1f} min)')
else:
    record('ace_smoke_gate', 'NOT_RUN', 'ACE not available in this environment')

In [ ]:
def ace_tables():
    rows = []
    for (s, v, sd), df in sorted(ACE_RES.items()):
        rows.append({'scene': s, 'gt': v, 'seed': sd, **summarise_errors(df), 'median_inliers': float(df.inliers.median()), 'min_inliers': float(df.inliers.min())})
    return pd.DataFrame(rows)

if ACE_READY and ACE_RES:
    load_finished_runs(); PER_SEED = ace_tables(); PER_SEED.to_csv(DIRS['results/tables'] / 'ace_per_seed.csv', index=False)
    print('ACE per-seed results on the official test sequences'); print(PER_SEED.round(2).to_string(index=False))
    agg = PER_SEED.groupby(['scene', 'gt']).agg(n_seeds=('seed', 'count'), acc_mean=('acc_5cm_5deg_pct', 'mean'), acc_std=('acc_5cm_5deg_pct', 'std'),
                                               median_t_cm_mean=('median_t_cm', 'mean'), median_r_deg_mean=('median_r_deg', 'mean')).reset_index()
    agg.to_csv(DIRS['results/tables'] / 'ace_seed_mean.csv', index=False); print('\nmean over the seeds that finished'); print(agg.round(2).to_string(index=False))
    # ---- parity with ACE's own scoring
    pt = max(v[0] for v in ACE_PARITY.values()); pr = max(v[1] for v in ACE_PARITY.values())
    record('ace_scoring_parity', 'PASS' if pt < CFG['ace']['parity_tol_t_cm'] and pr < CFG['ace']['parity_tol_r_deg'] else 'FAIL',
           f'max |our - ACE logged| over {len(ACE_PARITY)} runs: translation {pt:.4f} cm, rotation {pr:.4f} deg (tolerances {CFG["ace"]["parity_tol_t_cm"]} cm / {CFG["ace"]["parity_tol_r_deg"]} deg)')
    # ---- reproduction against published numbers (published values are NOT mixed into the measured tables)
    PUB_COL = 'published_acc_pct (NOT measured here)'; rep = []
    for s in CFG['scenes']:
        for v, key in [('sfm', 'ace_sfm_acc_5cm5deg'), ('dslam', 'ace_dslam_acc_5cm5deg')]:
            sub = PER_SEED[(PER_SEED.scene == s) & (PER_SEED['gt'] == v)]
            if len(sub) == 0: continue
            pub = PUBLISHED_REFERENCE[key][s]; m = float(sub.acc_5cm_5deg_pct.mean()); tol = CFG['ace']['repro_tol_pp'][s]
            rep.append({'scene': s, 'gt': v, 'seeds_done': len(sub), 'measured_acc_mean_pct': round(m, 2), 'published_acc_pct (NOT measured here)': pub,
                        'difference_pp': round(m - pub, 2), 'tolerance_pp': tol if v == 'sfm' else 'reported only (approximate third-party reference)',
                        'within_tolerance': (abs(m - pub) <= tol) if v == 'sfm' else None})
    REPRO = pd.DataFrame(rep); REPRO.to_csv(DIRS['results/tables'] / 'ace_reproduction.csv', index=False); print('\nReproduction check'); print(REPRO.to_string(index=False))
    for s in CFG['scenes']:
        r_ = REPRO[(REPRO.scene == s) & (REPRO['gt'] == 'sfm')]
        if len(r_) and int(r_.seeds_done.iloc[0]) == len(CFG['ace']['seeds_to_run']):
            record(f'ace_reproduction_sfm_{s}', 'PASS' if bool(r_.within_tolerance.iloc[0]) else 'FAIL',
                   f'{s}/SfM mean accuracy {float(r_.measured_acc_mean_pct.iloc[0]):.1f} % vs published {float(r_[PUB_COL].iloc[0]):.1f} % (tolerance {CFG["ace"]["repro_tol_pp"][s]:g} pp)')
        else: record(f'ace_reproduction_sfm_{s}', 'NOT_RUN', 'not all seeds finished yet')
    done_all = len(ACE_RES) == len(ACE_PLAN)
    record('ace_all_runs_complete', 'PASS' if done_all else 'NOT_RUN', f'{len(ACE_RES)}/{len(ACE_PLAN)} runs finished (2 scenes x 2 ground truths x {len(CFG["ace"]["seeds_to_run"])} seeds)')
    if REPRO[(REPRO['gt'] == 'sfm')].within_tolerance.eq(False).any():
        print('\nReproduction outside tolerance: this is reported, not tuned away or called expected. Differences to investigate: dataset preparation, ground-truth variant, pose conventions, training configuration (official environment is PyTorch 2.0/CUDA 11.8), random seed, evaluation implementation.')
else:
    for n in ['ace_scoring_parity', 'ace_reproduction_sfm_chess', 'ace_reproduction_sfm_stairs', 'ace_all_runs_complete']: record(n, 'NOT_RUN', 'no ACE results in this environment')

## 18. Evaluation pipeline: full-density comparison

*What:* the four methods are compared on the **official test sequences** with the full training map, for both ground-truth variants and every scene: **NetVLAD top-1**, **NetVLAD top-k** (k fixed on the development sequence), **ACE** (three seeds, per-seed statistics averaged; the seed spread is reported) and the **oracle nearest view** (the best any retrieval method could do: copy the pose of the closest training frame). For every method the notebook reports the median translation (cm) and rotation (°) error, the accuracy within 5 cm / 5°, the median and IQR per **equal-count distance bin** (5 bins on the nearest-view distance d(q)), the **usable range** (largest bin in which at least 50 % of the queries are usable; *censored* if every bin passes) and all of this again for the two **orientation strata** (nearest-view orientation difference ≤ 30° and > 30°).

*Why:* d(q) is the explanatory variable of the study. It is computed offline from ground-truth poses and never given to a method. Bins and strata are defined from d(q) and the orientation difference only, so every method is binned identically. Inside a stratum the 5 equal-count bins are formed within that stratum so that each bin keeps enough queries. No test frame is removed: every table is built from all test frames, which the code asserts.

*Expected output:* the comparison table, the per-bin table (CSV) and the usable-range table.

In [ ]:
FULL_METHODS = ['NetVLAD top-1', 'NetVLAD top-k', 'ACE', 'Oracle nearest view']
STRATA = ['all', 'orient<=30', 'orient>30']
MIN_STRATUM_N = 5 * 10          # a stratum needs at least 10 queries per bin to be binned

def usable_mask(df):
    th = CFG['thresholds']; return ((df.t_err_cm <= th['trans_cm']) & (df.r_err_deg <= th['rot_deg'])).values

def stratum_mask(orient, stratum):
    cut = CFG['orientation_split_deg']
    return np.ones(len(orient), bool) if stratum == 'all' else (orient <= cut if stratum == 'orient<=30' else orient > cut)

def oracle_table(map_ps, test_ps):
    orc_ps, _ = oracle_nearest_view(map_ps, test_ps); dd = nearest_view_distance(map_ps, test_ps)
    return frame_table(orc_ps, test_ps, {'d_nn_cm': dd['d_cm'], 'orient_nn_deg': dd['orient_deg']})

def collect_methods(nv1, nvk, ace_list, orc):
    return {'NetVLAD top-1': [nv1], 'NetVLAD top-k': [nvk], 'ACE': list(ace_list), 'Oracle nearest view': [orc]}

def binned_table(frames, d, mask, k=None):
    """Equal-count bins of d inside `mask`. frames: list of per-seed frame tables (same row order). Per-bin statistics are computed per seed and averaged."""
    k = k or CFG['distance_bins']; dm = d[mask]; b = assign_equal_count_bins(dm, k); rows = []
    for i in range(k):
        sel = b == i; per = []
        for df in frames:
            t, r, u = df.t_err_cm.values[mask][sel], df.r_err_deg.values[mask][sel], usable_mask(df)[mask][sel]
            per.append([np.median(t), np.percentile(t, 25), np.percentile(t, 75), np.median(r), np.percentile(r, 25), np.percentile(r, 75), 100 * u.mean()])
        per = np.array(per); m = per.mean(0)
        rows.append({'bin': i + 1, 'n': int(sel.sum()), 'd_lo_cm': float(dm[sel].min()), 'd_hi_cm': float(dm[sel].max()), 'd_median_cm': float(np.median(dm[sel])),
                     'median_t_cm': m[0], 'q25_t_cm': m[1], 'q75_t_cm': m[2], 'median_r_deg': m[3], 'q25_r_deg': m[4], 'q75_r_deg': m[5], 'usable_pct': m[6],
                     'median_t_cm_seed_std': float(per[:, 0].std(ddof=1)) if len(frames) > 1 else 0.0})
    return pd.DataFrame(rows)

def usable_range_summary(frames, d, mask):
    res = [usable_range(d[mask], usable_mask(df)[mask]) for df in frames]
    st = [r['status'] for r in res]; edges = [r['upper_edge'] for r in res if r['upper_edge'] is not None]
    status = st[0] if len(set(st)) == 1 else 'mixed'
    return {'ur_status': status, 'ur_upper_edge_cm': float(np.mean(edges)) if edges else np.nan, 'ur_per_seed': ';'.join(f"{r['status']}:{r['upper_edge'] if r['upper_edge'] is None else round(r['upper_edge'], 1)}" for r in res)}

def evaluate_methods(methods, tag_cols):
    """Returns (overall table, per-bin table) for a dict method -> list of frame tables, binned by d_nn_cm of that dict's own tables."""
    cmp_rows, bin_rows = [], []
    for name, frames in methods.items():
        if not frames: continue
        d = frames[0].d_nn_cm.values; o = frames[0].orient_nn_deg.values
        for f_ in frames: assert (f_.key.values == frames[0].key.values).all(), 'frame tables are not row-aligned'
        for st in STRATA:
            m = stratum_mask(o, st)
            if m.sum() < MIN_STRATUM_N: continue
            per = [summarise_errors(f_[m].reset_index(drop=True)) for f_ in frames]
            row = {**tag_cols, 'method': name, 'stratum': st, 'n_seeds': len(frames), 'n': int(m.sum()),
                   'median_t_cm': np.mean([p['median_t_cm'] for p in per]), 'median_r_deg': np.mean([p['median_r_deg'] for p in per]),
                   'acc_5cm5deg_pct': np.mean([p['acc_5cm_5deg_pct'] for p in per])}
            if len(frames) > 1:
                row.update({'median_t_cm_seed_std': np.std([p['median_t_cm'] for p in per], ddof=1), 'acc_seed_std': np.std([p['acc_5cm_5deg_pct'] for p in per], ddof=1)})
            row.update(usable_range_summary(frames, d, m)); cmp_rows.append(row)
            bt = binned_table(frames, d, m); 
            for c, v_ in {**tag_cols, 'method': name, 'stratum': st}.items(): bt[c] = v_
            bin_rows.append(bt)
    return pd.DataFrame(cmp_rows), pd.concat(bin_rows, ignore_index=True)

FULL = {}          # (scene, gt) -> {method: [frame tables]}
FULL_READY = bool(NV_READY and ACE_READY and ACE_RES and T1 and TOPK)
if FULL_READY:
    load_finished_runs()
    CMP, BINS = [], []
    for s in CFG['scenes']:
        for v in CFG['gt_variants']:
            sp = SPLITS[(s, v)]; seeds = sorted(sd for (ss, vv, sd) in ACE_RES if (ss, vv) == (s, v))
            if not seeds: continue
            FULL[(s, v)] = collect_methods(T1[(s, v)], TOPK[(s, v)], [ACE_RES[(s, v, sd)] for sd in seeds], oracle_table(sp['train'], sp['test']))
            c_, b_ = evaluate_methods(FULL[(s, v)], {'scene': s, 'gt': v}); CMP.append(c_); BINS.append(b_)
    CMP_DF = pd.concat(CMP, ignore_index=True); BIN_DF = pd.concat(BINS, ignore_index=True)
    CMP_DF.to_csv(DIRS['results/tables'] / 'comparison_full_density.csv', index=False); BIN_DF.to_csv(DIRS['results/tables'] / 'comparison_per_bin_full_density.csv', index=False)
    # ---- integrity checks of the evaluation pipeline
    n_ok = all(len(f_) == len(SPLITS[k_]['test']) and set(f_.key) == set(SPLITS[k_]['test'].keys) for k_, ms in FULL.items() for fl in ms.values() for f_ in fl)
    record('eval_no_frames_removed', 'PASS' if n_ok else 'FAIL', 'every method/seed table covers exactly the official test frames of its scene (no hard frame dropped)')
    bins_ok = all(np.ptp(np.bincount(assign_equal_count_bins(f['NetVLAD top-1'][0].d_nn_cm.values, CFG['distance_bins']))) <= 1 for f in FULL.values())
    orc_ok = all(np.abs(f['Oracle nearest view'][0].t_err_cm.values - f['Oracle nearest view'][0].d_nn_cm.values).max() < 1e-9 for f in FULL.values())
    same_d = all(np.allclose(fl[0].d_nn_cm.values, ms['NetVLAD top-1'][0].d_nn_cm.values) for ms in FULL.values() for fl in ms.values())
    record('eval_bins_and_oracle', 'PASS' if bins_ok and orc_ok and same_d else 'FAIL', f'equal-count bins differ by at most one query={bins_ok}; oracle error == d(q) on all frames={orc_ok}; all methods use the identical d(q)={same_d}')
    show = CMP_DF[CMP_DF.stratum == 'all'][['scene', 'gt', 'method', 'n_seeds', 'median_t_cm', 'median_r_deg', 'acc_5cm5deg_pct', 'ur_status', 'ur_upper_edge_cm']]
    print('Full-density comparison (all test queries; ACE = mean over seeds)'); print(show.round(2).to_string(index=False))
    print('\nUsable range per method (stratum = all). "censored" = every distance bin is usable, the range limit was not reached:')
    print(CMP_DF[CMP_DF.stratum == 'all'][['scene', 'gt', 'method', 'ur_status', 'ur_upper_edge_cm', 'ur_per_seed']].round(1).to_string(index=False))
    print('\nOrientation strata (median translation error in cm; n = queries in the stratum)')
    print(CMP_DF.pivot_table(index=['scene', 'gt', 'method'], columns='stratum', values='median_t_cm').round(2).to_string())
    print('\nShare of test queries per stratum:'); print(CMP_DF[CMP_DF.method == 'Oracle nearest view'][['scene', 'gt', 'stratum', 'n']].to_string(index=False))
else:
    CMP_DF = BIN_DF = pd.DataFrame()
    for n in ['eval_no_frames_removed', 'eval_bins_and_oracle']: record(n, 'NOT_RUN', 'needs the NetVLAD and ACE results')
    print('Full-density comparison not run: NetVLAD and/or ACE results are missing.')

## 19. Statistical analysis

*What:* for each scene, ground truth and method the **Spearman rank correlation** between the nearest-view distance d(q) and the translation error, with a **95 % block-bootstrap confidence interval** (2000 resamples). Consecutive video frames are almost duplicates, so frames are **not** independent; the bootstrap therefore resamples **blocks of consecutive frames** (50, or 100 if the development check below rejects 50; within a test sequence, in frame order) with replacement. All methods share the same resampled blocks, so differences between methods are **paired**. For ACE the statistic is the mean over the three seeds.

*Block length check (development sequence only):* the pre-declared length of 50 is checked on the **development** sequence with NetVLAD top-1 (queries = development sequence, map = development map). The width of the bootstrap interval of the Spearman correlation is computed for block lengths 1 to 200. Block resampling underestimates the uncertainty when blocks are shorter than the dependence length, so the interval width rises with the block length and then flattens. The rule fixed in advance: **length 50 is adequate if its interval width is at least 85 % of the widest width obtained with blocks of 100 or 200**. The test sequences are not used for this check. **Fallback fixed in advance:** if length 50 fails the rule on the development sequence, the longer block length **100** is used for all reported intervals instead (the decision uses development data only and is recorded; it is never revised after seeing test results). A sensitivity table with lengths 25, 50 and 100 is always printed.

*Hypotheses* (frozen in Section 1): **H1** NetVLAD top-1 Spearman ≥ 0.5 in both scenes; **H2** ACE has a lower correlation than NetVLAD top-1 in both scenes (paired block-bootstrap difference; "supported" needs the whole 95 % interval below zero, otherwise the verdict is "direction only" or "not supported"). H3 is evaluated in Section 20 because it needs the density experiments.

In [ ]:
BOOT = CFG['bootstrap']
from scipy.stats import rankdata

def make_blocks(ps, L):
    """Blocks of L consecutive frames inside each sequence (frame order); indices refer to the rows of ps."""
    out = []
    for q in sorted(set(ps.seq)):
        idx = np.where(ps.seq == q)[0]; idx = idx[np.argsort(ps.frame[idx])]
        out += [idx[i:i + L] for i in range(0, len(idx), L)]
    return out

def _spearman(x, y):
    rx, ry = rankdata(x), rankdata(y); rx -= rx.mean(); ry -= ry.mean()
    den = np.sqrt((rx ** 2).sum() * (ry ** 2).sum()); return float((rx * ry).sum() / den) if den > 0 else np.nan

def block_resample_matrix(blocks, fn, n_out, n_boot, seed):
    rng = np.random.default_rng(seed); nb = len(blocks); out = np.empty((n_boot, n_out))
    for b in range(n_boot): out[b] = fn(np.concatenate([blocks[i] for i in rng.integers(0, nb, nb)]))
    return out

def series_stats(d, series):
    """series: dict name -> list of (t_err, r_err) arrays per seed. Returns fn(idx) -> vector and the label list."""
    labels = [(n, s) for n in series for s in ['rho_t', 'rho_r', 'med_t', 'med_r', 'acc']]
    th = CFG['thresholds']
    def fn(idx):
        out = []
        for n, ser in series.items():
            v = np.array([[_spearman(d[idx], t[idx]), _spearman(d[idx], r[idx]), np.median(t[idx]), np.median(r[idx]),
                           100 * np.mean((t[idx] <= th['trans_cm']) & (r[idx] <= th['rot_deg']))] for t, r in ser]).mean(0)
            out.extend(v)
        return np.array(out)
    return fn, labels

def ci_of(draws, level=None):
    a = (1 - (level or BOOT['ci'])) / 2; return np.percentile(draws, [100 * a, 100 * (1 - a)], axis=0)

# ---------- block-length check on the DEVELOPMENT sequence (test frames untouched) ----------
BLOCK_CHECK = []
if NV_READY:
    for s in CFG['scenes']:
        for v in CFG['gt_variants']:
            sp = SPLITS[(s, v)]; pred, _, _ = netvlad_top1(s, sp['dev_map'], sp['dev']); e = score_predictions(pred, sp['dev'])
            d = nearest_view_distance(sp['dev_map'], sp['dev'])['d_cm']; t = e.t_err_cm.values; r_ = e.r_err_deg.values
            widths = {}
            for L in [1, 5, 10, 25, 50, 100, 200]:
                fn, _ = series_stats(d, {'m': [(t, r_)]}); dr = block_resample_matrix(make_blocks(sp['dev'], L), lambda i: fn(i)[:1], 1, 500, seed=11)
                lo, hi = ci_of(dr)[:, 0]; widths[L] = float(hi - lo); BLOCK_CHECK.append({'scene': s, 'gt': v, 'block_len': L, 'n_blocks': len(make_blocks(sp['dev'], L)), 'ci_width_spearman_t': widths[L]})
            # lag-k autocorrelation of the ranked translation error (descriptive support for the dependence length)
            rk = rankdata(t) - rankdata(t).mean(); acf = {lag: float((rk[:-lag] * rk[lag:]).sum() / (rk ** 2).sum()) for lag in [1, 10, 25, 50, 100]}
            BLOCK_CHECK[-1]['acf_lag1_10_25_50_100'] = ' / '.join(f'{acf[l]:.2f}' for l in [1, 10, 25, 50, 100])
    BLOCK_DF = pd.DataFrame(BLOCK_CHECK); BLOCK_DF.to_csv(DIRS['results/tables'] / 'bootstrap_block_length_check_dev.csv', index=False)
    print('Spearman 95 % CI width on the development sequence vs block length'); print(BLOCK_DF.pivot_table(index=['scene', 'gt'], columns='block_len', values='ci_width_spearman_t').round(3).to_string())
    ok_blk, det = True, []
    for (s, v), g in BLOCK_DF.groupby(['scene', 'gt']):
        w = g.set_index('block_len').ci_width_spearman_t; ratio = w[50] / max(w[100], w[200]); ok_blk &= ratio >= 0.85; det.append(f'{s}/{v}: width(50)/max(width(100), width(200)) = {ratio:.2f}')
    BLOCK_LEN = BOOT['block_len'] if ok_blk else 100
    record('bootstrap_block_length_dev', 'PASS', f'rule: ratio >= 0.85 -> ' + ('length 50 adequate; ' if ok_blk else 'length 50 NOT adequate on the development sequence, pre-declared fallback 100 used; ') + '; '.join(det))
    print(f'block length used for all reported intervals: {BLOCK_LEN}')
else:
    BLOCK_LEN = BOOT['block_len']; record('bootstrap_block_length_dev', 'NOT_RUN', 'no descriptors')

In [ ]:
STATS_ROWS, H_ROWS, SENS_ROWS = [], [], []
if FULL_READY:
    for (s, v), ms in FULL.items():
        test = SPLITS[(s, v)]['test']; d = ms['NetVLAD top-1'][0].d_nn_cm.values
        series = {n: [(f_.t_err_cm.values, f_.r_err_deg.values) for f_ in fl] for n, fl in ms.items() if n != 'Oracle nearest view'}
        fn, labels = series_stats(d, series); point = fn(np.arange(len(test)))
        draws = block_resample_matrix(make_blocks(test, BLOCK_LEN), fn, len(labels), BOOT['n_boot'], seed=2026)
        lo, hi = ci_of(draws); col = {lab: j for j, lab in enumerate(labels)}
        for (n, stat), j in col.items():
            STATS_ROWS.append({'scene': s, 'gt': v, 'method': n, 'statistic': stat, 'estimate': point[j], 'ci95_lo': lo[j], 'ci95_hi': hi[j], 'n_blocks': len(make_blocks(test, BLOCK_LEN))})
        # H1 and H2
        r1 = col[('NetVLAD top-1', 'rho_t')]; ra = col[('ACE', 'rho_t')]; diff = draws[:, ra] - draws[:, r1]; dlo, dhi = np.percentile(diff, [2.5, 97.5])
        H_ROWS.append({'scene': s, 'gt': v, 'rho_netvlad_top1': point[r1], 'rho_ace': point[ra], 'diff_ace_minus_netvlad': point[ra] - point[r1], 'diff_ci95_lo': dlo, 'diff_ci95_hi': dhi,
                       'H1_point_ge_0.5': bool(point[r1] >= 0.5), 'H1_ci_lo_ge_0.5': bool(lo[r1] >= 0.5),
                       'H2_verdict': 'supported' if dhi < 0 else ('direction only' if point[ra] < point[r1] else 'not supported')})
        # sensitivity to the block length (pre-declared 50 stays the reported one)
        for L in [25, 50, 100]:
            dr = block_resample_matrix(make_blocks(test, L), lambda i: fn(i)[[r1, ra]], 2, 500, seed=7); l_, h_ = ci_of(dr)
            SENS_ROWS.append({'scene': s, 'gt': v, 'block_len': L, 'netvlad_top1_rho_ci': f'[{l_[0]:.3f}, {h_[0]:.3f}]', 'ace_rho_ci': f'[{l_[1]:.3f}, {h_[1]:.3f}]'})
    STATS_DF = pd.DataFrame(STATS_ROWS); STATS_DF.to_csv(DIRS['results/tables'] / 'bootstrap_statistics_full_density.csv', index=False)
    H_DF = pd.DataFrame(H_ROWS); H_DF.to_csv(DIRS['results/tables'] / 'hypotheses_H1_H2.csv', index=False)
    print('Spearman correlation of translation error with d(q), 95 % block-bootstrap CI (block length per the development check, see above)')
    print(STATS_DF[STATS_DF.statistic == 'rho_t'].round(3).to_string(index=False))
    print('\nHypotheses H1 and H2'); print(H_DF.round(3).to_string(index=False))
    print('\nSensitivity to block length (CI of the Spearman correlation)'); print(pd.DataFrame(SENS_ROWS).to_string(index=False))
    H1_ALL = bool(H_DF['H1_point_ge_0.5'].all()); H2_ALL = bool((H_DF.H2_verdict == 'supported').all())
    record('statistics_bootstrap', 'PASS', f'block bootstrap (length {BLOCK_LEN}, {BOOT["n_boot"]} resamples, paired across methods) completed for {len(FULL)} scene/ground-truth combinations')
else:
    STATS_DF = H_DF = pd.DataFrame(); H1_ALL = H2_ALL = None
    record('statistics_bootstrap', 'NOT_RUN', 'needs the full-density comparison')

## 20. Training-density experiments

*What:* both methods are re-run with a **thinned training map** while the **test sequences stay unchanged**. Densities: **100 %** (Sections 17–18), **50 %** (uniform thinning: every 2nd frame of each training sequence), **25 %** and **10 %** (random thinning: a seeded random subset of each training sequence). The thinned frame set is identical for both ground-truth variants and every method (the pose files of the two variants contain the same frames). Thinned maps are subsets of the full training set, so NetVLAD descriptors are reused (no extra GPU extraction); ACE is **retrained from scratch** on each thinned map (official trainer, same hyper-parameters) and tested on the full test sequences.

*Rules:* (i) retrieval, the nearest-view distance d(q) and the oracle always use the **thinned** map, so a query's distance grows as the map gets sparser; (ii) the NetVLAD top-k neighbourhood size k is **re-tuned on the development sequence against an equally thinned development map**, never on test frames; (iii) ACE seeds: 3 seeds at 50 % (needed for H3), 1 seed (seed 0) at 25 % and 10 % to bound the GPU budget, which is stated wherever those numbers appear (the seed list is `CFG['densities']['seeds']`); (iv) 50 % uses uniform and 25 %/10 % use random thinning, as specified, so the 50 % → 25 % step also changes the thinning scheme (reported as a limitation); (v) finished ACE runs are stored and never repeated.

*H3 (frozen):* on Chess, from 100 % to 50 % density the NetVLAD top-1 median translation error grows by a larger factor than ACE's. It is evaluated with the paired block bootstrap of Section 19 on the ratio `median error at 50 % / median error at 100 %`.

In [ ]:
DENS = CFG['densities']; DENS_TAGS = {0.5: 'd50', 0.25: 'd25', 0.10: 'd10'}
DENS_DEVMAPS, DENS_FRAMES = {}, {}

def thin_random(ps, frac, seed):
    """Seeded random subset of each sequence (frame order kept); round(frac * n) frames per sequence."""
    keep = np.zeros(len(ps), dtype=bool); rng = np.random.default_rng(seed)
    for q in sorted(set(ps.seq)):
        idx = np.where(ps.seq == q)[0]; idx = idx[np.argsort(ps.frame[idx])]
        pick = np.sort(rng.choice(len(idx), max(1, int(round(frac * len(idx)))), replace=False)); keep[idx[pick]] = True
    return ps.subset(keep, ps.name + f'/random{frac}')

def thin_map(ps, frac, scene, salt=0):
    if frac == 0.5: return thin_uniform(ps, DENS['uniform_stride_50pct'])
    return thin_random(ps, frac, DENS['random_thinning_seed'] + 1000 * salt + int(round(frac * 1000)) + 7 * CFG['scenes'].index(scene))

for s in CFG['scenes']:
    for v in CFG['gt_variants']:
        sp = SPLITS[(s, v)]
        for frac, tag in DENS_TAGS.items():
            DENS_MAPS[(s, v, tag)] = thin_map(sp['train'], frac, s, 0); DENS_DEVMAPS[(s, v, tag)] = thin_map(sp['dev_map'], frac, s, 1)
# ---- validity of the thinned maps
ok_map, det = True, []
for s in CFG['scenes']:
    for frac, tag in DENS_TAGS.items():
        a, b = DENS_MAPS[(s, 'dslam', tag)], DENS_MAPS[(s, 'sfm', tag)]; full = SPLITS[(s, 'sfm')]['train']; test = SPLITS[(s, 'sfm')]['test']
        share = len(b) / len(full)
        good = set(a.keys) == set(b.keys) and set(b.keys) <= set(full.keys) and set(b.keys).isdisjoint(test.keys) and abs(share - frac) < 0.02 \
               and set(b.seq) == set(full.seq) and len(set(DENS_DEVMAPS[(s, 'sfm', tag)].keys) - set(SPLITS[(s, 'sfm')]['dev_map'].keys)) == 0
        ok_map &= bool(good); det.append(f'{s} {tag}: {len(b)}/{len(full)} frames ({100 * share:.1f} %)')
record('density_maps_valid', 'PASS' if ok_map else 'FAIL', 'thinned maps are subsets of the training set, identical for both variants, disjoint from test, within 2 pp of the target share, every sequence kept: ' + '; '.join(det))
print('\n'.join(det))

In [ ]:
def nv_density(s, v, tag):
    """NetVLAD top-1 / top-k / oracle on the thinned map (CPU only, descriptors cached)."""
    sp = SPLITS[(s, v)]; mp = DENS_MAPS[(s, v, tag)]; dmp = DENS_DEVMAPS[(s, v, tag)]
    pred, nn, sim = netvlad_top1(s, mp, sp['test']); dd = nearest_view_distance(mp, sp['test'])
    extra = {'d_nn_cm': dd['d_cm'], 'orient_nn_deg': dd['orient_deg']}
    t1 = frame_table(pred, sp['test'], {'retrieved_key': mp.keys[nn], 'similarity': sim, **extra})
    def _tune():
        rows = []
        for k in NV_CFG['k_grid']:
            if k > len(dmp): continue
            p_, _ = netvlad_topk(s, dmp, sp['dev'], k); e = score_predictions(p_, sp['dev'])
            rows.append({'k': k, 'dev_median_t_cm': float(e.t_err_cm.median()), 'dev_median_r_deg': float(e.r_err_deg.median())})
        return rows
    rows = cached_json(DIRS['cache'] / f'netvlad_dev_tuning_{s}_{v}_{tag}.json', _tune)
    k = int(pd.DataFrame(rows).sort_values(['dev_median_t_cm', 'k']).iloc[0].k)
    pk, info = netvlad_topk(s, mp, sp['test'], k)
    tk = frame_table(pk, sp['test'], {'k': k, 'retrieved_key': mp.keys[info['idx'][:, 0]], **extra})
    save_predictions(t1, f'netvlad_top1_{s}_{v}_{tag}'); save_predictions(tk, f'netvlad_topk_{s}_{v}_{tag}')
    return t1, tk, oracle_table(mp, sp['test']), k, pd.DataFrame(rows).assign(scene=s, gt=v, density=tag)

NV_DENS_K, NV_DENS_TUNE = {}, []
if NV_READY and FULL_READY:
    for s in CFG['scenes']:
        for v in CFG['gt_variants']:
            for tag in DENS_TAGS.values():
                t1, tk, orc, k, tune = nv_density(s, v, tag)
                DENS_FRAMES[(s, v, tag)] = {'NetVLAD top-1': [t1], 'NetVLAD top-k': [tk], 'Oracle nearest view': [orc], 'ACE': []}; NV_DENS_K[(s, v, tag)] = k; NV_DENS_TUNE.append(tune)
    pd.concat(NV_DENS_TUNE).to_csv(DIRS['results/tables'] / 'netvlad_topk_dev_tuning_thinned_maps.csv', index=False)
    print('k chosen on the thinned development maps:', {f'{a}/{b}/{c}': k for (a, b, c), k in NV_DENS_K.items()})
    if any(k == max(NV_CFG['k_grid']) for k in NV_DENS_K.values()): print('NOTE: some chosen k are at the upper edge of the pre-registered grid (not extended).')
    record('density_netvlad', 'PASS', 'NetVLAD top-1/top-k/oracle evaluated on every thinned map; k re-tuned on thinned development maps only')
else:
    record('density_netvlad', 'NOT_RUN', 'needs the NetVLAD and ACE results')

In [ ]:
def build_thinned_scene(s, v, tag):
    """ACE-layout folder with the thinned TRAIN frames (links to the full folder) and the unchanged TEST folder."""
    full, tgt, mp = ace_scene_dir(s, v), ace_scene_dir(s, v, tag), DENS_MAPS[(s, v, tag)]
    marker = tgt / '.thin_complete'
    if marker.exists() and marker.read_text() == str(len(mp)): return tgt
    for sub in ['rgb', 'poses', 'calibration']:
        d_ = tgt / 'train' / sub
        if d_.exists(): shutil.rmtree(d_)
        d_.mkdir(parents=True, exist_ok=True)
    for key in mp.keys:
        name = key.replace('/', '-')
        for sub, suf in [('rgb', '.color.png'), ('poses', '.pose.txt'), ('calibration', '.calibration.txt')]:
            src, dst = (full / 'train' / sub / (name + suf)).resolve(), tgt / 'train' / sub / (name + suf)
            try: os.symlink(src, dst)
            except OSError: shutil.copy(src, dst)
    if not (tgt / 'test').exists():
        try: os.symlink((full / 'test').resolve(), tgt / 'test', target_is_directory=True)
        except OSError: shutil.copytree(full / 'test', tgt / 'test')
    marker.write_text(str(len(mp))); return tgt

def check_thinned_scene(s, v, tag):
    tgt, mp = ace_scene_dir(s, v, tag), DENS_MAPS[(s, v, tag)]; names = lambda sub, suf: {p.name[:-len(suf)] for p in (tgt / 'train' / sub).iterdir() if p.name.endswith(suf)}
    exp = {k.replace('/', '-') for k in mp.keys}
    ok = names('rgb', '.color.png') == names('poses', '.pose.txt') == names('calibration', '.calibration.txt') == exp
    ntest = len(list((tgt / 'test' / 'rgb').iterdir())); return bool(ok and ntest == len(SPLITS[(s, v)]['test']))

DENS_PLAN = [(s, v, tag, sd) for frac, tag in DENS_TAGS.items() for s in CFG['scenes'] for v in CFG['gt_variants'] for sd in DENS['seeds'][frac]]
def load_finished_dens_runs():
    for (s, v, tag, sd) in DENS_PLAN:
        plog = run_dir(s, v, sd, tag) / f'poses_{ace_scene_dir(s, v, tag).name}_seed{sd}.txt'
        if (s, v, tag, sd) not in ACE_DENS_RES and plog.exists() and len(plog.read_text().splitlines()) == len(POSES[(s, v, 'test')]):
            df, pt, pr = score_ace_run(s, v, sd, plog, tag); ACE_DENS_RES[(s, v, tag, sd)] = df; ACE_DENS_PARITY[(s, v, tag, sd)] = (pt, pr)

DENS_BUDGET_MIN = 600       # wall-clock budget of this cell; every finished run is stored, so re-running the cell resumes
if ACE_READY and FULL_READY and REAL_IMAGES_AVAILABLE:
    ok_dirs = True
    for s in CFG['scenes']:
        for v in CFG['gt_variants']:
            for tag in DENS_TAGS.values(): build_thinned_scene(s, v, tag); ok_dirs &= check_thinned_scene(s, v, tag)
    ds = [run_json(DATASET_TEST, args=[ace_scene_dir(s, 'sfm', tag), 'train'], cwd=ACE_DIR, env=ace_env())['n'] for s in CFG['scenes'] for tag in DENS_TAGS.values()]
    ok_dirs &= ds == [len(DENS_MAPS[(s, 'sfm', tag)]) for s in CFG['scenes'] for tag in DENS_TAGS.values()]
    record('density_scene_folders', 'PASS' if ok_dirs else 'FAIL', 'thinned ACE folders hold exactly the thinned train frames (images = poses = calibration) and the unchanged test folder; ACE CamLocDataset reads the expected frame counts')
    load_finished_dens_runs()
    secs = [json.loads((run_dir(s_, v_, sd_) / 'run.json').read_text()).get('train_seconds') for (s_, v_, sd_) in ACE_PLAN if (run_dir(s_, v_, sd_) / 'run.json').exists()]
    mean_min = (np.mean(secs) / 60 + 1.5) if secs else 8.0
    todo = [r for r in DENS_PLAN if r not in ACE_DENS_RES]
    print(f'{len(DENS_PLAN)} density runs planned, {len(todo)} still to do; about {mean_min:.1f} min per run (train + test) -> roughly {len(todo) * mean_min / 60:.1f} GPU hours')
    t_start = time.time()
    for run in todo:
        if (time.time() - t_start) / 60 > DENS_BUDGET_MIN:
            print(f'time budget of {DENS_BUDGET_MIN} min used up; re-run this cell to resume ({len(ACE_DENS_RES)}/{len(DENS_PLAN)} done)'); break
        s, v, tag, sd = run; t0 = time.time(); df = run_and_score(s, v, sd, tag)
        print(f'{run}: acc {summarise_errors(df)["acc_5cm_5deg_pct"]:.1f} %  median {df.t_err_cm.median():.2f} cm / {df.r_err_deg.median():.2f} deg  ({(time.time() - t0) / 60:.1f} min)')
    for (s, v, tag, sd), df in sorted(ACE_DENS_RES.items()):
        if (s, v, tag) in DENS_FRAMES and df is not None and not any(df is x for x in DENS_FRAMES[(s, v, tag)]['ACE']): DENS_FRAMES[(s, v, tag)]['ACE'].append(df)
    pt = max([x[0] for x in ACE_DENS_PARITY.values()] or [0]); pr = max([x[1] for x in ACE_DENS_PARITY.values()] or [0])
    record('density_ace_scoring_parity', 'PASS' if ACE_DENS_PARITY and pt < CFG['ace']['parity_tol_t_cm'] and pr < CFG['ace']['parity_tol_r_deg'] else 'FAIL', f'max |our - ACE logged| over {len(ACE_DENS_PARITY)} density runs: {pt:.4f} cm, {pr:.4f} deg')
    record('density_ace_all_runs_complete', 'PASS' if len(ACE_DENS_RES) == len(DENS_PLAN) else 'NOT_RUN', f'{len(ACE_DENS_RES)}/{len(DENS_PLAN)} density runs finished')
else:
    for n in ['density_scene_folders', 'density_ace_scoring_parity', 'density_ace_all_runs_complete']: record(n, 'NOT_RUN', 'ACE results or real images not available in this environment')

In [ ]:
DENS_ROWS, DENS_BINS = [], []
if FULL_READY and DENS_FRAMES:
    ALL_DENS = {}
    for (s, v), ms in FULL.items(): ALL_DENS[(s, v, 'full')] = ms
    ALL_DENS.update(DENS_FRAMES)
    tag_frac = {'full': 1.0, **{t: f for f, t in DENS_TAGS.items()}}
    for (s, v, tag), ms in sorted(ALL_DENS.items(), key=lambda kv: (kv[0][0], kv[0][1], -tag_frac[kv[0][2]])):
        c_, b_ = evaluate_methods(ms, {'scene': s, 'gt': v, 'density': tag_frac[tag]}); c_['n_map'] = len(SPLITS[(s, v)]['train']) if tag == 'full' else len(DENS_MAPS[(s, v, tag)])
        c_['median_d_nn_cm'] = float(ms['NetVLAD top-1'][0].d_nn_cm.median()); DENS_ROWS.append(c_); DENS_BINS.append(b_)
    DENS_DF = pd.concat(DENS_ROWS, ignore_index=True); DENS_DF.to_csv(DIRS['results/tables'] / 'density_comparison.csv', index=False)
    pd.concat(DENS_BINS, ignore_index=True).to_csv(DIRS['results/tables'] / 'density_per_bin.csv', index=False)
    print('Training-density results (all test queries; ACE = mean over the seeds in n_seeds)')
    print(DENS_DF[DENS_DF.stratum == 'all'][['scene', 'gt', 'density', 'n_map', 'method', 'n_seeds', 'median_t_cm', 'median_r_deg', 'acc_5cm5deg_pct', 'median_d_nn_cm', 'ur_status', 'ur_upper_edge_cm']].round(2).to_string(index=False))
    # ---- H3: Chess, 100 % -> 50 %: growth factor of the median translation error, paired block bootstrap
    H3_ROWS = []
    for v in CFG['gt_variants']:
        for s in CFG['scenes']:
            if (s, v, 'd50') not in ALL_DENS: continue
            full_, thin_ = ALL_DENS[(s, v, 'full')], ALL_DENS[(s, v, 'd50')]; test = SPLITS[(s, v)]['test']
            sds_full = sorted(sd for (ss, vv, sd) in ACE_RES if (ss, vv) == (s, v)); sds_thin = sorted(sd for (ss, vv, tg, sd) in ACE_DENS_RES if (ss, vv, tg) == (s, v, 'd50')); sds = [x for x in sds_full if x in sds_thin]
            if not sds: continue
            ace_full = {sd: ACE_RES[(s, v, sd)] for sd in sds}; ace_thin = {sd: ACE_DENS_RES[(s, v, 'd50', sd)] for sd in sds}
            def fn(idx, full_=full_, thin_=thin_, ace_full=ace_full, ace_thin=ace_thin, sds=sds):
                nv_t = np.median(thin_['NetVLAD top-1'][0].t_err_cm.values[idx]) / np.median(full_['NetVLAD top-1'][0].t_err_cm.values[idx])
                nv_r = np.median(thin_['NetVLAD top-1'][0].r_err_deg.values[idx]) / np.median(full_['NetVLAD top-1'][0].r_err_deg.values[idx])
                ac_t = np.mean([np.median(ace_thin[sd].t_err_cm.values[idx]) / np.median(ace_full[sd].t_err_cm.values[idx]) for sd in sds])
                ac_r = np.mean([np.median(ace_thin[sd].r_err_deg.values[idx]) / np.median(ace_full[sd].r_err_deg.values[idx]) for sd in sds])
                return np.array([nv_t, ac_t, nv_t - ac_t, nv_r, ac_r])
            pt_ = fn(np.arange(len(test))); dr = block_resample_matrix(make_blocks(test, BLOCK_LEN), fn, 5, BOOT['n_boot'], seed=303); lo, hi = ci_of(dr)
            H3_ROWS.append({'scene': s, 'gt': v, 'ace_seeds_used': len(sds), 'growth_netvlad_top1_t': pt_[0], 'growth_ace_t': pt_[1], 'diff_netvlad_minus_ace': pt_[2],
                            'diff_ci95_lo': lo[2], 'diff_ci95_hi': hi[2], 'growth_netvlad_top1_r': pt_[3], 'growth_ace_r': pt_[4],
                            'verdict': ('supported' if lo[2] > 0 else 'direction only' if pt_[2] > 0 else 'not supported') if s == 'chess' else 'descriptive (H3 concerns Chess)'})
    H3_DF = pd.DataFrame(H3_ROWS); H3_DF.to_csv(DIRS['results/tables'] / 'hypothesis_H3.csv', index=False)
    print('\nH3: growth factor of the median error when the training density drops from 100 % to 50 %'); print(H3_DF.round(3).to_string(index=False))
    record('density_tables_and_H3', 'PASS' if len(H3_DF) else 'FAIL', f'{len(DENS_DF)} density table rows; H3 computed for {len(H3_DF)} scene/ground-truth combinations')
else:
    DENS_DF = H3_DF = pd.DataFrame(); ALL_DENS = {}
    record('density_tables_and_H3', 'NOT_RUN', 'needs the density runs')

## 21. Visualisation

*What:* every figure is drawn from the tables above and saved as **PNG (200 dpi) and PDF** in `results/figures/`.

1. **Error versus distance** (one figure per ground truth and error type): median error in the five equal-count distance bins, with the interquartile range as a band, for NetVLAD top-1, NetVLAD top-k, ACE and the oracle; one row per scene, columns *all queries*, *orientation ≤ 30°*, *orientation > 30°*. The x-axis is the median d(q) of each bin; the y-axis is logarithmic. ACE curves are the mean over seeds of the per-seed bin statistics.
2. **Usable share per bin**: the share of usable queries (≤ 5 cm and ≤ 5°) per distance bin, with the 50 % line that defines the usable range.
3. **Spearman correlation with 95 % block-bootstrap intervals** (Section 19).
4. **Training density**: median error against the share of training frames kept.

Colours are colour-blind safe and every method also has its own marker and line style, so no figure relies on colour alone.

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
STYLE = {'NetVLAD top-1': dict(color='#0072B2', marker='o', ls='-'), 'NetVLAD top-k': dict(color='#009E73', marker='s', ls='--'),
         'ACE': dict(color='#D55E00', marker='^', ls='-'), 'Oracle nearest view': dict(color='#444444', marker='x', ls=':')}
GT_LABEL = {'dslam': 'D-SLAM ground truth', 'sfm': 'SfM pseudo ground truth'}
FIG_DIR = DIRS['results/figures']; FIG_FILES = []

def save_fig(fig, name):
    for ext in ['png', 'pdf']:
        p = FIG_DIR / f'{name}.{ext}'; fig.savefig(p, dpi=200, bbox_inches='tight'); FIG_FILES.append(p)

def fig_error_vs_distance(gt, metric):
    col, ylabel = ('median_t_cm', 'translation error (cm)') if metric == 't' else ('median_r_deg', 'rotation error (deg)')
    lo_c, hi_c = (col.replace('median', 'q25'), col.replace('median', 'q75'))
    fig, axs = plt.subplots(len(CFG['scenes']), len(STRATA), figsize=(14, 4.6 * len(CFG['scenes'])), squeeze=False)
    for i, s in enumerate(CFG['scenes']):
        for j, st in enumerate(STRATA):
            ax = axs[i, j]; sub = BIN_DF[(BIN_DF.scene == s) & (BIN_DF['gt'] == gt) & (BIN_DF.stratum == st)]
            for m in FULL_METHODS:
                g = sub[sub.method == m].sort_values('bin')
                if not len(g): continue
                ax.plot(g.d_median_cm, g[col], label=m, lw=1.8, ms=5, **STYLE[m])
                if m != 'Oracle nearest view': ax.fill_between(g.d_median_cm, g[lo_c], g[hi_c], color=STYLE[m]['color'], alpha=0.13, lw=0)
            if metric == 't': ax.axhline(CFG['thresholds']['trans_cm'], color='k', lw=0.8, alpha=0.5)
            else: ax.axhline(CFG['thresholds']['rot_deg'], color='k', lw=0.8, alpha=0.5)
            ax.set_yscale('log'); ax.set_xlabel('distance to nearest training view d(q) (cm, bin median)'); ax.set_ylabel(ylabel)
            n_ = int(sub[sub.method == 'NetVLAD top-1'].n.sum()); ax.set_title(f'{s.capitalize()} - {st.replace("orient", "orientation ")} (n = {n_})', fontsize=10); ax.grid(alpha=0.25, which='both')
            if i == 0 and j == 0: ax.legend(fontsize=8, frameon=False)
    fig.suptitle(f'Median {ylabel.split(" (")[0]} versus distance, band = interquartile range - {GT_LABEL[gt]}', y=1.0, fontsize=12); fig.tight_layout(); return fig

def fig_usable_share(gt):
    fig, axs = plt.subplots(1, len(CFG['scenes']), figsize=(12, 4.2), squeeze=False)
    for i, s in enumerate(CFG['scenes']):
        ax = axs[0, i]; sub = BIN_DF[(BIN_DF.scene == s) & (BIN_DF['gt'] == gt) & (BIN_DF.stratum == 'all')]
        for m in FULL_METHODS:
            g = sub[sub.method == m].sort_values('bin'); ax.plot(g.d_median_cm, g.usable_pct, label=m, lw=1.8, ms=5, **STYLE[m])
        ax.axhline(100 * CFG['usable_share'], color='k', lw=0.8, ls='--'); ax.set_ylim(-3, 103); ax.set_xlabel('distance to nearest training view d(q) (cm, bin median)'); ax.set_ylabel('usable queries (%)')
        ax.set_title(f'{s.capitalize()} - {GT_LABEL[gt]}', fontsize=10); ax.grid(alpha=0.25)
        if i == 0: ax.legend(fontsize=8, frameon=False)
    fig.tight_layout(); return fig

if FULL_READY and len(BIN_DF):
    for gt in CFG['gt_variants']:
        for metric in ['t', 'r']:
            f_ = fig_error_vs_distance(gt, metric); save_fig(f_, f'error_vs_distance_{"translation" if metric == "t" else "rotation"}_{gt}'); plt.show()
        f_ = fig_usable_share(gt); save_fig(f_, f'usable_share_per_bin_{gt}'); plt.show()
    # ---- Spearman forest plot
    sp_ = STATS_DF[STATS_DF.statistic == 'rho_t']; fig, axs = plt.subplots(1, len(CFG['gt_variants']), figsize=(12, 4), squeeze=False)
    for j, gt in enumerate(CFG['gt_variants']):
        ax = axs[0, j]; rows_ = [(s, m) for s in CFG['scenes'] for m in ['NetVLAD top-1', 'NetVLAD top-k', 'ACE']]
        for y, (s, m) in enumerate(rows_):
            r = sp_[(sp_.scene == s) & (sp_['gt'] == gt) & (sp_.method == m)].iloc[0]
            ax.errorbar(r.estimate, y, xerr=[[r.estimate - r.ci95_lo], [r.ci95_hi - r.estimate]], fmt=STYLE[m]['marker'], color=STYLE[m]['color'], capsize=3)
        ax.axvline(0.5, color='k', lw=0.8, ls='--'); ax.set_yticks(range(len(rows_))); ax.set_yticklabels([f'{s.capitalize()} - {m}' for s, m in rows_], fontsize=8); ax.invert_yaxis()
        ax.set_xlabel('Spearman correlation(d(q), translation error), 95 % block-bootstrap CI'); ax.set_title(GT_LABEL[gt], fontsize=10); ax.grid(alpha=0.25, axis='x')
    fig.tight_layout(); save_fig(fig, 'spearman_correlation_ci'); plt.show()
    # ---- training density
    if len(DENS_DF):
        fig, axs = plt.subplots(len(CFG['gt_variants']), len(CFG['scenes']), figsize=(11, 8), squeeze=False)
        for i, gt in enumerate(CFG['gt_variants']):
            for j, s in enumerate(CFG['scenes']):
                ax = axs[i, j]; sub = DENS_DF[(DENS_DF.scene == s) & (DENS_DF['gt'] == gt) & (DENS_DF.stratum == 'all')]
                for m in FULL_METHODS:
                    g = sub[sub.method == m].sort_values('density')
                    if not len(g): continue
                    yerr = g['median_t_cm_seed_std'].fillna(0).values if 'median_t_cm_seed_std' in g else None
                    ax.errorbar(100 * g.density, g.median_t_cm, yerr=yerr, label=m, lw=1.8, ms=5, capsize=2, **STYLE[m])
                ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xticks([10, 25, 50, 100]); ax.set_xticklabels(['10', '25', '50', '100']); ax.invert_xaxis()
                ax.set_xlabel('training frames kept (%)'); ax.set_ylabel('median translation error (cm)'); ax.set_title(f'{s.capitalize()} - {GT_LABEL[gt]}', fontsize=10); ax.grid(alpha=0.25, which='both')
                if i == 0 and j == 0: ax.legend(fontsize=8, frameon=False)
        fig.suptitle('Median translation error versus training density (error bars: ACE seed std where more than one seed)', fontsize=11); fig.tight_layout(); save_fig(fig, 'error_vs_training_density'); plt.show()
    ok_fig = all(p.exists() and p.stat().st_size > 5000 for p in FIG_FILES)
    record('figures_saved', 'PASS' if ok_fig else 'FAIL', f'{len(FIG_FILES)} figure files (PNG and PDF) written to results/figures')
else:
    record('figures_saved', 'NOT_RUN', 'needs the full-density comparison')

## 22. Failure analysis

*What:* for **each method** (NetVLAD top-1, NetVLAD top-k, ACE) and **each scene** the **20 worst test frames** by translation error (ties by rotation error), on the SfM pseudo ground truth (the reference ACE is normally reported on; the D-SLAM overlap is printed). ACE frames are ranked by their mean error over the seeds. For every frame the table gives the errors, d(q), the orientation difference to the nearest view, a retrieval confidence (NetVLAD similarity of the retrieved image; ACE inlier count) and an image-sharpness value (variance of the Laplacian of the grey image). The **image evidence** is a montage with the query, the retrieved training image (NetVLAD) and the nearest training view by ground truth.

*Tags (descriptive, rule-based, thresholds fixed in advance, not causal proof).* A worst frame is tagged **far-from-map** if its d(q) is above the 90th percentile of all test queries of the scene, **rotated-view** if the orientation difference to the nearest view exceeds 30°, **blurry** if its sharpness is below the 10th percentile of the scene's test frames and **low-confidence** if the confidence value is below the 10th percentile of the method's own confidence on that scene. A frame with no tag is **untagged**. To make the tags interpretable, each tag's share among the worst 20 is printed next to its share among *all* test frames (the base rate); a tag is only informative if it is clearly more frequent among the worst frames than in general.

In [ ]:
FAIL_GT, N_WORST = 'sfm', 20
def sharpness(path):
    from PIL import Image
    g = np.asarray(Image.open(path).convert('L'), dtype=np.float64)
    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    return float(lap.var())

def test_sharpness(scene):
    p = DIRS['cache'] / f'sharpness_{scene}_test.json'
    def _calc(): return {k: sharpness(image_path(scene, k, 'test')) for k in SPLITS[(scene, FAIL_GT)]['test'].keys}
    return cached_json(p, _calc)

def montage(scene, method, worst, fname):
    from PIL import Image
    n = len(worst); cols = 4; rows = int(np.ceil(n / cols)); fig, axs = plt.subplots(rows, cols, figsize=(4 * cols * 1.25, rows * 1.9 * 1.25), squeeze=False)
    for ax in axs.ravel(): ax.axis('off')
    for ax, (_, r) in zip(axs.ravel(), worst.iterrows()):
        panels = [Image.open(image_path(scene, r.key, 'test')).convert('RGB').resize((160, 120))]
        if isinstance(r.get('retrieved_key'), str): panels.append(Image.open(image_path(scene, r.retrieved_key, 'train')).convert('RGB').resize((160, 120)))
        panels.append(Image.open(image_path(scene, r.nearest_key, 'train')).convert('RGB').resize((160, 120)))
        canvas = np.concatenate([np.asarray(p) for p in panels], axis=1); ax.imshow(canvas); ax.axis('off')
        ax.set_title(f"#{int(r['rank'])} {r.key}  {r.t_err_cm:.0f} cm / {r.r_err_deg:.0f} deg  d={r.d_nn_cm:.0f} cm  [{r.tags}]", fontsize=6.5)
    cap = 'panels: query | retrieved image (NetVLAD only) | nearest training view by ground truth'
    fig.suptitle(f'{scene.capitalize()} - {method}: 20 worst test frames ({GT_LABEL[FAIL_GT]}); {cap}', fontsize=9); fig.tight_layout(); fig.savefig(FIG_DIR / f'{fname}.png', dpi=130, bbox_inches='tight'); FIG_FILES.append(FIG_DIR / f'{fname}.png'); return fig

FAIL_TABLES, FAIL_TAGSTATS, FAIL_OVERLAP = [], [], []
if FULL_READY and REAL_IMAGES_AVAILABLE:
    for s in CFG['scenes']:
        sp = SPLITS[(s, FAIL_GT)]; sharp = test_sharpness(s); sh = np.array([sharp[k] for k in sp['test'].keys])
        d_all = FULL[(s, FAIL_GT)]['NetVLAD top-1'][0]; nearest = sp['train'].keys[nearest_view_distance(sp['train'], sp['test'])['nn_index']]
        per_method_worst = {}
        for m in ['NetVLAD top-1', 'NetVLAD top-k', 'ACE']:
            fl = FULL[(s, FAIL_GT)][m]; base = fl[0].copy()
            base['t_err_cm'] = np.mean([f_.t_err_cm.values for f_ in fl], 0); base['r_err_deg'] = np.mean([f_.r_err_deg.values for f_ in fl], 0)
            conf_name, conf = ('similarity', fl[0].similarity.values) if m == 'NetVLAD top-1' else ('inliers', np.mean([f_.inliers.values for f_ in fl], 0)) if m == 'ACE' else (None, None)
            if m == 'NetVLAD top-k':
                conf_name, conf = 'similarity', T1[(s, FAIL_GT)].similarity.values       # confidence of the top-1 neighbour of the same query
            tbl = pd.DataFrame({'key': base.key.values, 't_err_cm': base.t_err_cm.values, 'r_err_deg': base.r_err_deg.values, 'd_nn_cm': base.d_nn_cm.values,
                                'orient_nn_deg': base.orient_nn_deg.values, 'confidence_name': conf_name, 'confidence': conf, 'sharpness': sh, 'nearest_key': nearest})
            if 'retrieved_key' in base: tbl['retrieved_key'] = base.retrieved_key.values
            elif m == 'NetVLAD top-k': tbl['retrieved_key'] = T1[(s, FAIL_GT)].retrieved_key.values
            thr = {'far': np.percentile(tbl.d_nn_cm, 90), 'blur': np.percentile(tbl.sharpness, 10), 'conf': np.percentile(tbl.confidence, 10)}
            tag_arr = {'far-from-map': (tbl.d_nn_cm > thr['far']).values, 'rotated-view': (tbl.orient_nn_deg > CFG['orientation_split_deg']).values,
                       'blurry': (tbl.sharpness < thr['blur']).values, 'low-confidence': (tbl.confidence < thr['conf']).values}
            tbl['tags'] = [','.join(t for t in tag_arr if tag_arr[t][i]) or 'untagged' for i in range(len(tbl))]
            order = np.lexsort((-tbl.r_err_deg.values, -tbl.t_err_cm.values)); worst = tbl.iloc[order[:N_WORST]].copy(); worst.insert(0, 'rank', np.arange(1, N_WORST + 1))
            worst.insert(0, 'method', m); worst.insert(0, 'scene', s); FAIL_TABLES.append(worst); per_method_worst[m] = set(worst.key)
            for t in tag_arr:
                FAIL_TAGSTATS.append({'scene': s, 'method': m, 'tag': t, 'share_among_worst20_pct': 100 * worst.tags.str.contains(t).mean(), 'share_among_all_test_frames_pct': 100 * tag_arr[t].mean()})
            FAIL_TAGSTATS.append({'scene': s, 'method': m, 'tag': 'untagged', 'share_among_worst20_pct': 100 * (worst.tags == 'untagged').mean(), 'share_among_all_test_frames_pct': 100 * np.mean([x == 'untagged' for x in tbl.tags])})
            save_predictions(worst, f'failure_worst20_{m.replace(" ", "_").replace("-", "")}_{s}_{FAIL_GT}')
            montage(s, m, worst, f'failure_montage_{m.replace(" ", "_").replace("-", "")}_{s}'); plt.show()
        names = list(per_method_worst)
        for a_ in range(len(names)):
            for b_ in range(a_ + 1, len(names)): FAIL_OVERLAP.append({'scene': s, 'pair': f'{names[a_]} / {names[b_]}', 'shared_of_20': len(per_method_worst[names[a_]] & per_method_worst[names[b_]])})
    FAIL_DF = pd.concat(FAIL_TABLES, ignore_index=True); FAIL_DF.to_csv(DIRS['results/tables'] / 'failure_worst20_all.csv', index=False)
    TAG_DF = pd.DataFrame(FAIL_TAGSTATS); TAG_DF.to_csv(DIRS['results/tables'] / 'failure_tag_statistics.csv', index=False)
    print('Tag frequency among the 20 worst frames versus all test frames (percent)'); print(TAG_DF.round(1).to_string(index=False))
    print('\nOverlap of the worst-20 sets between methods'); print(pd.DataFrame(FAIL_OVERLAP).to_string(index=False))
    # ground-truth sensitivity: how many of the SfM worst-20 are also in the D-SLAM worst-20 (same method, same scene)
    ov = []
    for s in CFG['scenes']:
        for m in ['NetVLAD top-1', 'NetVLAD top-k']:
            w_sfm = set(FAIL_DF[(FAIL_DF.scene == s) & (FAIL_DF.method == m)].key); fl = FULL[(s, 'dslam')][m][0]
            w_ds = set(fl.sort_values(['t_err_cm', 'r_err_deg'], ascending=False).key.iloc[:N_WORST]); ov.append({'scene': s, 'method': m, 'shared_with_dslam_worst20': len(w_sfm & w_ds)})
    print('\nOverlap with the D-SLAM worst-20 (retrieval methods; same predictions, different reference poses)'); print(pd.DataFrame(ov).to_string(index=False))
    record('failure_analysis', 'PASS' if len(FAIL_DF) == 3 * len(CFG['scenes']) * N_WORST else 'FAIL', f'{len(FAIL_DF)} worst frames listed (3 methods x {len(CFG["scenes"])} scenes x {N_WORST}) with image montages')
else:
    FAIL_DF = TAG_DF = pd.DataFrame(); record('failure_analysis', 'NOT_RUN', 'needs the results and the real images')

In [ ]:
# Gap between the two ground-truth variants on the SAME frames (pose-only measurement; why both variants are evaluated)
GAP_ROWS = []
for s in CFG['scenes']:
    for sp in ['train', 'test']:
        a, b = POSES[(s, 'dslam', sp)], POSES[(s, 'sfm', sp)]
        b = b.by_key(a.keys); te = translation_error_cm(a.centres, b.centres); re_ = rotation_error_deg(a.R, b.R)
        GAP_ROWS.append({'scene': s, 'split': sp, 'n': len(a), 'median_t_gap_cm': np.median(te), 'p90_t_gap_cm': np.percentile(te, 90),
                         'median_r_gap_deg': np.median(re_), 'share_gap_gt_5cm_pct': 100 * np.mean(te > 5)})
GAP_DF = pd.DataFrame(GAP_ROWS); print(GAP_DF.round(3).to_string(index=False))

## 23. Result tables

*What:* the measured tables are written to `results/tables/` and printed below. Three groups are kept apart: (1) **pose-only references** (controls that use no image: mean-pose floor, shuffled poses, oracle nearest view), (2) the **localisation results** printed in Sections 18–20, (3) **published reference values**, labelled as *not measured here*.

In [ ]:
POSE_REF_DF = pd.DataFrame(POSE_REF_ROWS); DIST_DF = pd.DataFrame(DIST_ROWS); INV_DF = pd.DataFrame(SPLIT_ROWS)
T = DIRS['results/tables']
POSE_REF_DF.to_csv(T / 'pose_only_references.csv', index=False); DIST_DF.to_csv(T / 'distance_distribution.csv', index=False)
GAP_DF.to_csv(T / 'gt_variant_gap.csv', index=False); INV_DF.to_csv(T / 'split_structures.csv', index=False)
print('Pose-only references (controls, NOT localisation methods)'); print(POSE_REF_DF.round(2).to_string(index=False))
print('\nDistance of queries to the map'); print(DIST_DF.round(2).to_string(index=False))
if FULL_READY:
    main = CMP_DF[CMP_DF.stratum == 'all'].copy()
    def fmt(r, c, sd): return f'{r[c]:.2f}' + (f' +- {r[sd]:.2f}' if sd in r and pd.notna(r.get(sd)) else '')
    MAIN_DF = pd.DataFrame({'scene': main.scene, 'ground truth': main['gt'], 'method': main.method,
                            'median t (cm)': [fmt(r, 'median_t_cm', 'median_t_cm_seed_std') for _, r in main.iterrows()], 'median R (deg)': main.median_r_deg.round(2),
                            'acc 5cm/5deg (%)': [fmt(r, 'acc_5cm5deg_pct', 'acc_seed_std') for _, r in main.iterrows()],
                            'usable range': [('censored (all bins usable)' if r.ur_status == 'censored' else 'none' if r.ur_status == 'none' else f'{r.ur_upper_edge_cm:.0f} cm' + (' (mixed over seeds)' if r.ur_status == 'mixed' else '')) for _, r in main.iterrows()]})
    MAIN_DF.to_csv(T / 'main_results_table.csv', index=False); print('\nMain results (test sequences, full training density; ACE: mean +- std over seeds)'); print(MAIN_DF.to_string(index=False))
REF_DF = pd.DataFrame([{'quantity': k, 'chess': v['chess'], 'stairs': v['stairs'], 'source': v['source'], 'status': 'PUBLISHED, NOT MEASURED HERE'} for k, v in PUBLISHED_REFERENCE.items()])
REF_DF.to_csv(T / 'published_reference_values.csv', index=False); print('\nPublished references (kept apart from measurements)'); print(REF_DF.to_string(index=False))

## 24. Discussion

The findings below are **generated from the measured tables** of this run, so they always match the numbers above; nothing is typed in by hand. The limitations that follow are fixed facts about the design.

In [ ]:
from IPython.display import Markdown, display
def _g(df, **kw):
    m = np.ones(len(df), bool)
    for k, v in kw.items(): m &= (df[k] == v).values
    return df[m]
lines = []
if FULL_READY:
    for v in CFG['gt_variants']:
        for s in CFG['scenes']:
            sub = _g(CMP_DF, scene=s, gt=v, stratum='all'); r = {m: _g(sub, method=m).iloc[0] for m in FULL_METHODS}
            lines.append(f"* **{s.capitalize()}, {GT_LABEL[v]}:** median translation / rotation error - NetVLAD top-1 {r['NetVLAD top-1'].median_t_cm:.1f} cm / {r['NetVLAD top-1'].median_r_deg:.1f} deg, "
                         f"NetVLAD top-k {r['NetVLAD top-k'].median_t_cm:.1f} cm / {r['NetVLAD top-k'].median_r_deg:.1f} deg, ACE {r['ACE'].median_t_cm:.2f} cm / {r['ACE'].median_r_deg:.2f} deg (3-seed mean), "
                         f"oracle nearest view {r['Oracle nearest view'].median_t_cm:.1f} cm / {r['Oracle nearest view'].median_r_deg:.1f} deg. Accuracy within 5 cm / 5 deg: {r['NetVLAD top-1'].acc_5cm5deg_pct:.1f} %, {r['NetVLAD top-k'].acc_5cm5deg_pct:.1f} %, {r['ACE'].acc_5cm5deg_pct:.1f} %, {r['Oracle nearest view'].acc_5cm5deg_pct:.1f} %.")
            for m in ['NetVLAD top-1', 'NetVLAD top-k', 'ACE']:
                u = r[m]; lines.append(f"  * usable range of {m}: " + ('**censored** - every distance bin is usable, so the limit lies beyond the largest d(q) in the test set' if u.ur_status == 'censored' else 'no distance bin reaches 50 % usable queries' if u.ur_status == 'none' else f'up to about {u.ur_upper_edge_cm:.0f} cm' + (' (seeds disagree)' if u.ur_status == 'mixed' else '')))
    for _, h in H_DF.iterrows():
        lines.append(f"* **H1/H2, {h.scene.capitalize()}, {GT_LABEL[h['gt']]}:** Spearman(d(q), translation error) NetVLAD top-1 = {h.rho_netvlad_top1:.2f} (H1 point estimate >= 0.5: {h['H1_point_ge_0.5']}; whole 95 % CI >= 0.5: {h['H1_ci_lo_ge_0.5']}); ACE = {h.rho_ace:.2f}; difference ACE - NetVLAD = {h.diff_ace_minus_netvlad:.2f} (95 % CI {h.diff_ci95_lo:.2f} to {h.diff_ci95_hi:.2f}) -> H2 **{h.H2_verdict}**.")
    lines.append(f"* **H1 overall (both scenes, both ground truths, point estimate):** {'supported' if H1_ALL else 'not supported in every case'}. **H2 overall (interval entirely below zero in every case):** {'supported' if H2_ALL else 'not supported in every case - see the per-case verdicts'}.")
    for _, h in H3_DF.iterrows():
        lines.append(f"* **H3, {h.scene.capitalize()}, {GT_LABEL[h['gt']]}, 100 % -> 50 % density:** median translation error grows by x{h.growth_netvlad_top1_t:.2f} for NetVLAD top-1 and x{h.growth_ace_t:.2f} for ACE (difference {h.diff_netvlad_minus_ace:.2f}, 95 % CI {h.diff_ci95_lo:.2f} to {h.diff_ci95_hi:.2f}; {int(h.ace_seeds_used)} ACE seeds) -> **{h.verdict}**.")
    if len(DENS_DF):
        for s in CFG['scenes']:
            sub = _g(DENS_DF, scene=s, gt='sfm', stratum='all')
            txt = []
            for m in ['NetVLAD top-1', 'ACE']:
                g = _g(sub, method=m).sort_values('density', ascending=False); txt.append(f"{m}: " + ', '.join(f'{100 * d:.0f} % -> {t:.2f} cm' for d, t in zip(g.density, g.median_t_cm)))
            lines.append(f"* **Density trend, {s.capitalize()}, {GT_LABEL['sfm']} (median translation error):** " + '; '.join(txt) + '.')
    if len(FAIL_DF):
        for s in CFG['scenes']:
            for m in ['NetVLAD top-1', 'NetVLAD top-k', 'ACE']:
                t = _g(TAG_DF, scene=s, method=m); top = t[t.tag != 'untagged'].assign(lift=lambda x: x.share_among_worst20_pct - x.share_among_all_test_frames_pct).sort_values('lift', ascending=False).iloc[0]
                lines.append(f"* **Failure analysis, {s.capitalize()}, {m}:** the most over-represented tag among the 20 worst frames is *{top.tag}* ({top.share_among_worst20_pct:.0f} % of the worst frames versus {top.share_among_all_test_frames_pct:.0f} % of all test frames); {_g(t, tag='untagged').iloc[0].share_among_worst20_pct:.0f} % of the worst frames carry no tag.")
display(Markdown('\n'.join(lines) if lines else 'Results are not available in this environment, so no findings are generated.'))

**Limitations and caveats (fixed by the design)**

* The test split is the official one (two sequences for Chess and Stairs each in the standard 7-Scenes protocol); the block bootstrap treats blocks of consecutive frames (length chosen on the development sequence, Section 19) as independent, which is an approximation. Its adequacy rule on the development sequence and a sensitivity table are printed in Section 19.
* The two ground-truth variants differ by centimetres and a fraction of a degree (Section 7 table); the reported ACE accuracies therefore depend on the variant, and published numbers are only comparable when the variant matches. The D-SLAM reference numbers are a third-party approximation, not an official benchmark value.
* NetVLAD top-k uses a pre-declared grid for k. If a chosen k sits at the upper edge of the grid (printed in Sections 14 and 20), the best k may be larger; the grid was not extended after results were seen.
* The NetVLAD top-k interpolation weights are constrained only to sum to one, so they can be negative; the translation uses the signed weights and the rotation uses the non-negative part (Section 14). Other interpolation schemes could behave differently.
* ACE is trained and tested with the official code at its default settings; three small, documented patches (seed, step counter for current PyTorch, build against system OpenCV) are listed in Section 15 and saved as a diff. GPU non-determinism means a re-run may differ slightly from the stored seeds.
* Density experiments: 50 % uses uniform thinning and 25 %/10 % use random thinning, so the 50 % -> 25 % step also changes the thinning scheme. ACE uses 3 seeds at 50 % but only seed 0 at 25 % and 10 % (compute budget), so those two densities carry no seed spread. A thinned map keeps the same sequences, so it is sparser along each trajectory but not smaller in spatial extent.
* The nearest-view distance d(q) measures distance to the closest mapped *camera centre*; it ignores viewing direction, which is why the orientation strata are reported separately.
* The failure-analysis tags are descriptive and rule-based; they show association with a property of the frame, not its cause.

In [ ]:
CRITERIA = {
 'dataset downloaded/loaded (real images)':          ['image_download', 'real_images_loaded_and_matched'],
 'both pose variants loaded and conventions verified': ['pose_files_present', 'pose_conventions', 'ace_read_pose_parity', 'd_to_rgb_parity'],
 'D-SLAM variant unambiguous (calibration cross-check)': ['dslam_calibration_crosscheck'],
 'archive extraction policy':                          ['archive_extraction_policy'],
 'metrics validated':                                  ['metrics_vs_scipy', 'usable_range_logic', 'control3_synthetic_pose'],
 'nearest-view distance validated':                    ['distance_hand_example', 'distance_bruteforce_real', 'localiser_interface_image_only'],
 'train/dev/test structures, no leakage':              ['split_structures_no_leakage'],
 'negative controls 1-5':                              ['control1_mean_pose_floor', 'control2_shuffled_pose', 'control3_synthetic_pose', 'control4_oracle_bound', 'control5_pose_level', 'control5_loader_fixture', 'variant_keys_identical'],
 'NetVLAD features and top-1/top-k run':             ['netvlad_extraction', 'netvlad_descriptor_sanity', 'netvlad_topk_unit_tests', 'netvlad_topk_k1_equals_top1', 'netvlad_topk_dev_only_tuning'],
 'NetVLAD checks (self-retrieval, oracle bound, context range)': ['netvlad_self_retrieval', 'netvlad_oracle_bound', 'netvlad_sanity_range'],
 'ACE built and verified':                           ['ace_build', 'ace_solver_synthetic', 'ace_dataset_parity'],
 'ACE trained and tested, both GTs, all seeds':      ['ace_smoke_gate', 'ace_all_runs_complete'],
 'ACE scoring parity with its own per-frame output': ['ace_scoring_parity'],
 'ACE reproduction targets checked':                 ['ace_reproduction_sfm_chess', 'ace_reproduction_sfm_stairs'],
 'full-density comparison, bins and usable range':   ['eval_no_frames_removed', 'eval_bins_and_oracle'],
 'statistics (block length check, bootstrap)':       ['bootstrap_block_length_dev', 'statistics_bootstrap'],
 'training-density experiments':                     ['density_maps_valid', 'density_netvlad', 'density_scene_folders', 'density_ace_scoring_parity', 'density_ace_all_runs_complete', 'density_tables_and_H3'],
 'figures and failure analysis':                     ['figures_saved', 'failure_analysis'],
}
rows_, all_pass = [], True
for crit, names in CRITERIA.items():
    sts = [STATUS.get(n, {'status': 'NOT_RUN'})['status'] for n in names]
    st = 'FAIL' if 'FAIL' in sts else 'NOT_RUN' if 'NOT_RUN' in sts else 'PASS'
    all_pass &= st == 'PASS'; rows_.append({'criterion': crit, 'status': st, 'checks': ', '.join(f'{n}={s}' for n, s in zip(names, sts))})
CRIT_DF = pd.DataFrame(rows_); print(CRIT_DF[['criterion', 'status']].to_string(index=False))
ALL_CHECKS_PASS = bool(all_pass)
CRIT_DF.to_csv(DIRS['results/tables'] / 'completion_checks.csv', index=False)
(ROOT / 'status.json').write_text(json.dumps({'all_checks_pass': ALL_CHECKS_PASS, 'status': STATUS, 'config': CFG}, indent=1, default=str))
print('\nALL CHECKS PASS' if ALL_CHECKS_PASS else '\nSOME CHECKS ARE NOT PASS: see the table above. NOT_RUN means the step did not execute here; it is not a pass.')

## 25. Final summary

The cell above evaluates every recorded check against its criterion. **A criterion is satisfied only if all of its checks are `PASS`**; `NOT_RUN` means a step could not execute in this environment and is *not* a pass. The next cell packs all tables, figures, per-frame predictions and logs into one archive for download.

In [ ]:
import zipfile
res_root = DIRS['results/tables'].parent
ZIP = res_root.parent / 'p22_results.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(res_root.rglob('*')):
        if p.is_file(): z.write(p, p.relative_to(res_root.parent))
    z.write(ROOT / 'status.json', 'status.json')
print('results archive:', ZIP, f'({ZIP.stat().st_size / 1e6:.1f} MB);', len(list(res_root.rglob("*.csv"))), 'CSV files,', len(FIG_FILES) if 'FIG_FILES' in globals() else 0, 'figure files')
print('trained ACE heads are kept under', DIRS['runs'], '(not in the archive)')